# Intervention Gating
Configuration: `stages/02_intervention_gating/config`. Figures remain local; datasets are under `experiment/data/`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
intervention_gating_experiment_dir = ROOT / 'experiment/research/osd/stages/02_intervention_gating'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

intervention_gating_data_paths = ExperimentPaths(ROOT, intervention_gating_experiment_dir / 'config/paths.json')


# Two-stage BPOSD intervention gating and snapshot selection

This notebook separates intervention gating from conditional snapshot and OSD-order selection. Internal BP trajectories are generated exclusively by this notebook's
`trace_min_sum`. Phase 0 additionally calls the underlying `min_sum_trace` directly, solely to verify the endpoint of the previous
backend; it then compares the legacy raw-signed and current absolute-reliability
column orderings. The notebook does not call `CythonBPOSD`.
`standard_BPOSD` matches the project implementation: when BP does not converge, full OSD uses the raw signed LLRs;
absolute-reliability OSD is used only for intervention recovery.

The statistical unit is always the physical shot. Models, calibration, thresholds and the Stage-2 selector
are determined exclusively from development data; the final LER is reported only on held-out unbiased test shots.



In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from itertools import combinations, product
from math import comb
from pathlib import Path
from time import perf_counter
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.special import expit, logit
from scipy.stats import kendalltau
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython.cython_core.bp_core import min_sum_trace
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate

intervention_gating_config = json.loads((intervention_gating_experiment_dir / 'config/two_stage_bposd_intervention_snapshot_experiments.json').read_text(encoding='utf-8'))
if intervention_gating_config['optional_d9_generalization'] and 3 not in intervention_gating_config['levels']:
    intervention_gating_config['levels'].append(3)

if intervention_gating_config['smoke_mode']:
    intervention_gating_config.update(
        levels=[2],
        p_values=[0.05],
        minimum_shots_per_point=60,
        maximum_shots_per_point=60,
        sampling_batch_size=60,
        minimum_unsafe_events=1,
        reconciliation_shots_per_point=30,
        bp_max_iterations=8,
        reference_audit_shots_per_point=4,
        fixed_snapshot_iterations=[1, 2, 4, 8],
        feature_windows=[3],
        primary_window=3,
        top_k_values=[5],
        primary_top_k=5,
        gate_thresholds=[0.5, 0.8, 0.95],
        utility_penalties=[2, 10],
        primary_utility_penalty=10,
        prefix_buffer_sizes=[1, 3, -1],
        adaptive_order_thresholds=[0.5, 0.8],
        calibration_bins=4,
        bootstrap_repetitions=50,
        temporal_shuffle_repetitions=2,
        scaling_values=[1.0],
        damping_values=[0.0],
        parameter_scan_case_limit=6,
    )
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# levels: values restricted to {2, 3}.
# levels: excludes 1.
# p_values: each item is in (0, 1).
# Expected relationship: minimum_shots_per_point <= maximum_shots_per_point.
# full_osd_combination_sweep: 'cpp_combination_sweep'.
# primary_window: chosen from feature_windows.
# primary_top_k: chosen from top_k_values.
# primary_utility_penalty: chosen from utility_penalties.
# Expected relationship: development_fit_fraction + development_calibration_fraction < 1.
# stage2_selector_fit_fraction: in (0, 1).
intervention_gating_name = 'two_stage_bposd_intervention_snapshot_smoke' if intervention_gating_config['smoke_mode'] else 'two_stage_bposd_intervention_snapshot'
intervention_gating_out = intervention_gating_data_paths.run(smoke_mode=intervention_gating_config['smoke_mode'])

from decoders.cython.cython_core.osd_core import OSD_VERSION, osd_candidate_count
from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(intervention_gating_data_paths, intervention_gating_config, {'osd': OSD_VERSION})

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(intervention_gating_data_paths, intervention_gating_config)

intervention_gating_conf, intervention_gating_stats, intervention_gating_trace_dir = (intervention_gating_data_paths.output('config'), intervention_gating_data_paths.output('stats'), intervention_gating_data_paths.output('trace'))

for intervention_gating_directory in [intervention_gating_conf, intervention_gating_stats, intervention_gating_trace_dir]:
    intervention_gating_directory.mkdir(parents=True, exist_ok=True)

intervention_gating_config_path = intervention_gating_data_paths.output('config_json')
if not intervention_gating_config_path.is_file() or not intervention_gating_config['reuse_existing_data']:
    intervention_gating_config_path.write_text(json.dumps({'config': intervention_gating_config}, indent=2), encoding='utf-8')

from experiment.utils import misc

def intervention_gating_log(level, payload):
    return misc.log(level, payload, CONFIG=intervention_gating_config)

def intervention_gating_stage_cached(paths):
    return misc.stage_cached(paths, CONFIG=intervention_gating_config)

intervention_gating_save_csv = misc.save_csv

def intervention_gating_read_csv(path):
    assert path.is_file(), path
    return pd.read_csv(path, dtype={
        'shot_id': str,
        'syndrome': str,
        'truth': str,
        'error': str,
        'legacy_correction': str,
        'legacy_same_snapshot_correction': str,
        'legacy_unified_correction': str,
        'current_correction': str,
    }, low_memory=False)

def intervention_gating_split_for(shot_id):
    value = int.from_bytes(sha256(f"{intervention_gating_config['random_seed']}:{shot_id}".encode()).digest()[:8], 'little') / 2 ** 64
    fit_end = intervention_gating_config['development_fit_fraction']
    calibration_end = fit_end + intervention_gating_config['development_calibration_fraction']

    return 'fit' if value < fit_end else 'calibration' if value < calibration_end else 'test'

def stage2_role_for(shot_id, split):
    if split != 'calibration':
        return split
    value = int.from_bytes(sha256(f"{intervention_gating_config['random_seed']}:stage2:{shot_id}".encode()).digest()[:8], 'little') / 2 ** 64

    return 'selector_fit' if value < intervention_gating_config['stage2_selector_fit_fraction'] else 'policy_tune'

sns.set_theme(style='whitegrid')
intervention_gating_log(2, {'output': str(intervention_gating_out)})

assert intervention_gating_config['full_osd_combination_sweep'] == 'cpp_combination_sweep'


## Instrumented BP and audited OSD references

Initialize the native BP and audited OSD references first. Comparable PCM inputs, endpoint LLRs, ordering conventions and search budgets are necessary before attributing differences to gating.


In [ ]:
from experiment.utils import osd_reliability

intervention_gating_Context = osd_reliability.Context

from experiment.utils import gap_mechanisms

intervention_gating_BPTrace = gap_mechanisms.BPTrace
CONTEXTS = {}
intervention_gating_gf2_rank = osd_reliability.gf2_rank

def intervention_gating_context_for(level, p):
    key = (int(level), float(p))
    if key in CONTEXTS:
        return CONTEXTS[key]
    code = CA_9_2_5() if level == 2 else TriangularCode(level)
    circuit = CircuitBuilder(code).get_circuit_level_circuit(ErrorRate(p=float(p), p_meas=0), d_rounds=1)
    dem = circuit.detector_error_model()
    matrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = matrices.check_matrix.toarray().astype(np.uint8) % 2
    L = matrices.observables_matrix.toarray().astype(np.uint8) % 2
    priors = np.asarray(matrices.priors, dtype=float)
    checks, variables = np.nonzero(H)
    checks = checks.astype(np.int32)
    variables = variables.astype(np.int32)
    edge_ids = np.arange(len(checks), dtype=np.int32)
    check_edges = tuple((edge_ids[checks == c][::-1] for c in range(H.shape[0])))
    variable_edges = tuple((edge_ids[variables == v][::-1] for v in range(H.shape[1])))
    distance = 5 if level == 2 else int(code.d)
    assert distance in {5, 9} and intervention_gating_gf2_rank(H) == H.shape[0]
    order = intervention_gating_config['full_osd_order_by_distance'][str(distance)]
    assert osd_candidate_count(H.shape[1] - H.shape[0], order) == intervention_gating_config['full_osd_candidate_budget_by_distance'][str(distance)]
    result = intervention_gating_Context(level, distance, float(p), H, L, priors, np.log((1 - priors) / priors), dem, checks, variables, check_edges, variable_edges)
    CONTEXTS[key] = result

    return result

def intervention_gating_trace_min_sum(
    context,
    syndrome,
    iterations,
    scaling=1.0,
    damping=0.0,
):
    trace = context.bp_decoder.trace(
        syndrome, iterations, scaling=scaling, damping=damping,
        include_initial_valid=False, combine_scaling_damping=True,
    )
    return intervention_gating_BPTrace(trace.c2v, trace.v2c, trace.llr, trace.hard, trace.valid, trace.runtime_seconds, trace.first_valid)

from experiment.utils import misc

intervention_gating_bits = misc.bits_variant_2

def intervention_gating_bit_array(value, length):
    text = str(value).strip().zfill(length)
    assert len(text) == length and set(text) <= {'0', '1'}
    return np.fromiter((int(x) for x in text), dtype=np.uint8, count=length)

def intervention_gating_outcome(
    context,
    correction,
    syndrome,
    truth,
    error=None,
):
    correction = np.asarray(correction, dtype=np.uint8)
    valid = np.array_equal(context.H @ correction % 2, syndrome)
    label = context.L @ correction % 2
    correct = valid and np.array_equal(label, truth)
    exact = np.nan if error is None else int(np.array_equal(correction, error))

    return (int(valid), int(correct), exact, intervention_gating_bits(label))

def stable_rank_proxy(values):
    order = np.lexsort((np.arange(len(values)), np.asarray(values, dtype=float)))
    proxy = np.empty(len(values), dtype=float)
    proxy[order] = np.arange(len(values), dtype=float)

    return (proxy, order)

def pivot_columns(H_ordered):
    work = np.asarray(H_ordered, dtype=np.uint8).copy() % 2
    row = 0
    pivots = []

    for column in range(work.shape[1]):
        candidates = np.flatnonzero(work[row:, column])
        if not len(candidates):
            continue
        pivot = row + int(candidates[0])
        work[[row, pivot]] = work[[pivot, row]]

        for other in np.flatnonzero(work[:, column]):
            if other != row:
                work[other, column:] ^= work[row, column:]

        pivots.append(column)
        row += 1
        if row == work.shape[0]:
            break

    return np.asarray(pivots, dtype=int)

def gf2_inverse(matrix):
    work = np.asarray(matrix, dtype=np.uint8).copy() % 2
    inverse = np.eye(work.shape[0], dtype=np.uint8)
    for column in range(work.shape[0]):
        candidates = np.flatnonzero(work[column:, column])
        assert len(candidates), 'singular GF(2) pivot matrix'
        pivot = column + int(candidates[0])
        work[[column, pivot]] = work[[pivot, column]]
        inverse[[column, pivot]] = inverse[[pivot, column]]

        for row in np.flatnonzero(work[:, column]):
            if row != column:
                work[row] ^= work[column]
                inverse[row] ^= inverse[column]

    assert np.array_equal(np.asarray(matrix, dtype=np.uint8) @ inverse % 2, np.eye(work.shape[0], dtype=np.uint8))
    return inverse

def python_osd_reference(
    context,
    syndrome,
    column_order,
    order,
):
    H_ordered = context.H[:, column_order]
    pivots = pivot_columns(H_ordered)
    nonpivots = np.asarray([i for i in range(H_ordered.shape[1]) if i not in set(pivots)], dtype=int)
    inverse = gf2_inverse(H_ordered[:, pivots])
    e0 = inverse @ syndrome % 2
    best = np.zeros(H_ordered.shape[1], dtype=np.uint8)
    best[pivots] = e0
    weights = np.log(1.0 / context.priors)
    best_cost = float(sum(weights[i] * best[np.argsort(column_order)[i]] for i in range(len(best))))

    patterns = [] if order == 0 else [(i,) for i in range(len(nonpivots))] + list(combinations(range(min(int(order), len(nonpivots))), 2))
    for selected in patterns:
        candidate = np.zeros(H_ordered.shape[1], dtype=np.uint8)
        candidate[nonpivots[list(selected)]] = 1
        candidate[pivots] = (e0 + inverse @ (H_ordered[:, nonpivots] @ candidate[nonpivots] % 2)) % 2
        inverse_order = np.argsort(column_order)
        cost = float(sum(weights[i] * candidate[inverse_order[i]] for i in range(len(candidate))))
        if cost < best_cost:
            best = candidate
            best_cost = cost

    correction = np.empty_like(best)
    correction[column_order] = best
    assert np.array_equal(context.H @ correction % 2, syndrome)

    return correction

def osd_decode(
    context,
    syndrome,
    llr,
    order,
    convention='benchmark_signed',
    audit_reference=False,
):
    if convention in {'legacy_raw_signed', 'benchmark_signed'}:
        backend_input = np.asarray(llr, dtype=float)
        column_order = np.argsort(backend_input, kind='stable')
    else:
        assert convention == 'absolute'
        backend_input, column_order = stable_rank_proxy(np.abs(llr))

    started = perf_counter()
    correction = ordered_statistics_postprocess(np.ascontiguousarray(backend_input, dtype=np.float64), np.ascontiguousarray(syndrome, dtype=np.uint8), np.ascontiguousarray(context.H, dtype=np.uint8), int(order), np.ascontiguousarray(context.priors, dtype=np.float64))
    runtime = perf_counter() - started
    pivots = pivot_columns(context.H[:, column_order])
    assert len(pivots) == context.H.shape[0] and np.array_equal(context.H @ correction % 2, syndrome)

    if audit_reference:
        reference = python_osd_reference(context, syndrome, column_order, order)
        assert np.array_equal(correction, reference), f'OSD backend/reference mismatch for {convention}'

    return (correction, runtime, column_order, pivots)

def intervention_gating_full_order(context):
    return osd_reliability.full_order(context, CONFIG=intervention_gating_config)

def intervention_gating_candidate_count(context, order):
    return osd_candidate_count(context.H.shape[1] - context.H.shape[0], order)

intervention_gating_endpoint_of = osd_reliability.endpoint_of

def legacy_backend_endpoint(context, syndrome):
    """Isolated Phase-0 audit of the BP endpoint used by the previous backend."""

    m, n = context.H.shape
    edges = int(context.H.sum())
    msg = np.zeros(2 * edges, dtype=np.float64)
    head = np.full(m + n, -1, dtype=np.int32)
    head_nxt = np.full(2 * edges, -1, dtype=np.int32)
    to = np.full(2 * edges, -1, dtype=np.int32)
    edge = 0

    for check in range(m):
        for variable in np.flatnonzero(context.H[check]):
            to[edge] = variable + m
            head_nxt[edge] = head[check]
            head[check] = edge
            edge += 1
            to[edge] = check
            head_nxt[edge] = head[variable + m]
            head[variable + m] = edge
            edge += 1

    assert edge == 2 * edges
    llr, history = min_sum_trace(np.ascontiguousarray(syndrome, dtype=np.uint8), msg, head, head_nxt, to, m, n, np.ascontiguousarray(context.prior_llr, dtype=np.float64), intervention_gating_config['bp_max_iterations'], 1.0, True)
    endpoint = int(len(history))
    assert 1 <= endpoint <= intervention_gating_config['bp_max_iterations']

    return (np.asarray(llr), endpoint, np.asarray(history))

def intervention_gating_tesseract_predictions(context, syndromes):
    decoder = Tesseract(beam=intervention_gating_config['tesseract_beam'], pq_limit=intervention_gating_config['tesseract_pq_limit'], seed=intervention_gating_config['random_seed']).compile_decoder_for_dem(dem=context.dem)
    packed = np.packbits(np.asarray(syndromes, dtype=np.uint8), axis=1, bitorder='little')
    started = perf_counter()
    predictions = decoder.decode_shots_bit_packed(bit_packed_detection_event_data=packed)
    elapsed = (perf_counter() - started) / max(1, len(syndromes))

    return (np.unpackbits(predictions, axis=1, count=context.L.shape[0], bitorder='little').astype(np.uint8), elapsed)


## Experiments 1–3: reconciliation, unbiased intervention population and endpoint features

Reconcile the references, construct the population and compute endpoint features. This separates a baseline-semantics difference from a genuine opportunity to improve the intervention decision.


In [ ]:
POPULATION_PATH = intervention_gating_data_paths.output('unbiased_intervention_cases_csv')
RECON_PATH = intervention_gating_data_paths.output('forced_osd_reconciliation_trials_csv')
MISMATCH_PATH = intervention_gating_data_paths.output('forced_osd_mismatches_csv')

def intervention_gating_slope(values):
    values = np.asarray(values, dtype=float)
    return 0.0 if len(values) < 2 else float(np.polyfit(np.arange(len(values)), values, 1)[0])

def least_set(llr, k):
    return set(np.argsort(np.abs(llr), kind='stable')[:min(k, len(llr))].tolist())

def causal_features(
    context,
    trace,
    syndrome,
    t,
):
    assert 1 <= t < len(trace.llr)
    scale = float(np.median(np.abs(context.prior_llr))) + 1e-12
    residual = np.mean(np.abs(np.diff(trace.c2v[:t + 1], axis=0)), axis=1) / scale
    belief = np.mean(np.abs(np.diff(trace.llr[:t + 1], axis=0)), axis=1) / scale
    flips = np.mean(np.sign(trace.c2v[1:t + 1]) != np.sign(trace.c2v[:t]), axis=1)
    conflict = np.mean(trace.c2v[1:t + 1] * trace.v2c[1:t + 1] < 0, axis=1)
    current = np.abs(trace.llr[t]) / scale
    previous = np.abs(trace.llr[t - 1]) / scale
    tau = kendalltau(current, previous).statistic
    tau_history = [kendalltau(np.abs(trace.llr[u]), np.abs(trace.llr[u - 1])).statistic for u in range(1, t + 1)]
    tau_history = [0.0 if np.isnan(value) else float(value) for value in tau_history]
    k = min(intervention_gating_config['primary_top_k'], len(current))
    now = least_set(current, k)
    before = least_set(previous, k)
    ranks_now = np.empty(len(current), dtype=int)
    ranks_now[np.argsort(current, kind='stable')] = np.arange(len(current))
    ranks_before = np.empty(len(current), dtype=int)
    ranks_before[np.argsort(previous, kind='stable')] = np.arange(len(current))
    boundary = np.flatnonzero((ranks_now >= max(0, k - 2)) & (ranks_now < min(len(current), k + 2)))
    row = {
        'iteration': t,
        'syndrome_consistent': int(trace.valid[t]),
        'syndrome_residual': int(np.sum(context.H @ trace.hard[t] % 2 ^ syndrome)),
        'llr_min': float(current.min()),
        'llr_q10': float(np.quantile(current, 0.1)),
        'llr_q25': float(np.quantile(current, 0.25)),
        'llr_median': float(np.median(current)),
        'llr_max': float(current.max()),
        'llr_spread': float(current.max() - current.min()),
        'hard_prior_cost': float(np.sum(-np.log(np.where(trace.hard[t], context.priors, 1 - context.priors)))),
        'rank_kendall': 0.0 if np.isnan(tau) else float(tau),
        'rank_kendall_trend': intervention_gating_slope(tau_history),
        'rank_jaccard': len(now & before) / max(1, len(now | before)),
        'rank_boundary_motion': float(np.mean(np.abs(ranks_now[boundary] - ranks_before[boundary]))) if len(boundary) else 0.0,
        'message_residual': float(residual[-1]),
        'belief_movement': float(belief[-1]),
        'sign_flip_rate': float(flips[-1]),
        'extrinsic_conflict': float(conflict[-1]),
    }

    for window in intervention_gating_config['feature_windows']:
        width = min(window, len(residual))
        history = [least_set(trace.llr[u], k) for u in range(max(1, t - width + 1), t + 1)]
        row[f'residual_mean_w{window}'] = float(np.mean(residual[-width:]))
        row[f'residual_slope_w{window}'] = intervention_gating_slope(residual[-width:])
        row[f'belief_mean_w{window}'] = float(np.mean(belief[-width:]))
        row[f'flip_mean_w{window}'] = float(np.mean(flips[-width:]))
        row[f'conflict_mean_w{window}'] = float(np.mean(conflict[-width:]))
        row[f'rank_persistence_w{window}'] = float(np.mean([len(now & item) / k for item in history]))

    row.update(
        history_min_residual=float(np.min(residual)),
        history_max_rank_stability=max([len(least_set(trace.llr[u], k) & least_set(trace.llr[u - 1], k)) / k for u in range(1, t + 1)]),
        history_large_oscillation_episodes=int(np.sum(residual > 1.0)),
        history_valid_entries=int(np.sum(trace.valid[1:t + 1])),
        history_time_since_first_valid=-1 if trace.first_valid is None or t < trace.first_valid else t - trace.first_valid,
    )
    return row

def decoded_shot(
    context,
    syndrome,
    truth,
    error,
    tess_prediction,
    shot_id,
    tess_runtime,
    audit_reference=False,
    audit_legacy_backend=False,
):
    trace = intervention_gating_trace_min_sum(context, syndrome, intervention_gating_config['bp_max_iterations'])
    endpoint = intervention_gating_endpoint_of(trace)
    order = intervention_gating_full_order(context)
    current, current_time, current_order, current_pivots = osd_decode(context, syndrome, trace.llr[endpoint], order, 'benchmark_signed', audit_reference)

    if audit_legacy_backend:
        legacy_llr, legacy_endpoint, legacy_history = legacy_backend_endpoint(context, syndrome)
        bp_endpoint_equal = int(legacy_endpoint == endpoint)
        bp_llr_allclose = int(np.allclose(legacy_llr, trace.llr[endpoint], rtol=1e-12, atol=1e-12))
        bp_llr_exact = int(np.array_equal(legacy_llr, trace.llr[endpoint]))
        bp_llr_max_abs_diff = float(np.max(np.abs(legacy_llr - trace.llr[endpoint])))
        legacy, legacy_time, legacy_order, legacy_pivots = osd_decode(context, syndrome, legacy_llr, order, 'legacy_raw_signed', audit_reference)
        legacy_same_snapshot, _, _, _ = osd_decode(context, syndrome, trace.llr[endpoint], order, 'legacy_raw_signed', audit_reference)
        legacy_unified, _, legacy_unified_order, legacy_unified_pivots = osd_decode(context, syndrome, legacy_llr, order, 'benchmark_signed', audit_reference)
    else:
        legacy_llr, legacy_endpoint, legacy_history = (trace.llr[endpoint], endpoint, np.empty((0, context.H.shape[1])))
        bp_endpoint_equal = bp_llr_allclose = bp_llr_exact = -1
        bp_llr_max_abs_diff = np.nan

        if trace.valid[endpoint]:
            legacy = legacy_same_snapshot = legacy_unified = current
            legacy_order = legacy_unified_order = current_order
            legacy_pivots = legacy_unified_pivots = current_pivots
            legacy_time = 0.0
        else:
            legacy, legacy_time, legacy_order, legacy_pivots = osd_decode(context, syndrome, legacy_llr, order, 'legacy_raw_signed')
            legacy_same_snapshot = legacy
            legacy_unified = current
            legacy_unified_order = current_order
            legacy_unified_pivots = current_pivots

    legacy_unified_out = intervention_gating_outcome(context, legacy_unified, syndrome, truth, error)
    final0, final0_time, _, _ = osd_decode(context, syndrome, trace.llr[-1], 0)
    final1, final1_time, _, _ = osd_decode(context, syndrome, trace.llr[-1], 1)
    finalfull, finalfull_time, _, _ = osd_decode(context, syndrome, trace.llr[-1], order)
    endpoint0, endpoint0_time, _, _ = osd_decode(context, syndrome, trace.llr[endpoint], 0)
    endpoint1, endpoint1_time, _, _ = osd_decode(context, syndrome, trace.llr[endpoint], 1)

    if trace.valid[endpoint]:
        standard = trace.hard[endpoint]
        standard_osd_time = 0.0
    else:
        standard = legacy
        standard_osd_time = legacy_time

    standard_out = intervention_gating_outcome(context, standard, syndrome, truth, error)
    current_out = intervention_gating_outcome(context, current, syndrome, truth, error)
    legacy_out = intervention_gating_outcome(context, legacy, syndrome, truth, error)
    endpoint0_out = intervention_gating_outcome(context, endpoint0, syndrome, truth, error)
    endpoint1_out = intervention_gating_outcome(context, endpoint1, syndrome, truth, error)
    final0_out = intervention_gating_outcome(context, final0, syndrome, truth, error)
    final1_out = intervention_gating_outcome(context, final1, syndrome, truth, error)
    finalfull_out = intervention_gating_outcome(context, finalfull, syndrome, truth, error)
    features = causal_features(context, trace, syndrome, endpoint)
    row = {
        'shot_id': shot_id,
        'level': context.level,
        'distance': context.distance,
        'p': context.p,
        'split': intervention_gating_split_for(shot_id),
        'syndrome': intervention_gating_bits(syndrome),
        'truth': intervention_gating_bits(truth),
        'error': intervention_gating_bits(error),
        'error_weight': int(error.sum()),
        'endpoint': endpoint,
        'first_valid': -1 if trace.first_valid is None else trace.first_valid,
        'standard_correct': standard_out[1],
        'standard_exact': standard_out[2],
        'legacy_full_correct': legacy_out[1],
        'current_full_correct': current_out[1],
        'endpoint_osd0_correct': endpoint0_out[1],
        'endpoint_osd1_correct': endpoint1_out[1],
        'final_osd0_correct': final0_out[1],
        'final_osd1_correct': final1_out[1],
        'final_full_correct': finalfull_out[1],
        'tesseract_correct': int(np.array_equal(tess_prediction, truth)),
        'standard_runtime': float(trace.runtime[endpoint] + standard_osd_time),
        'endpoint_bp_runtime': float(trace.runtime[endpoint]),
        'endpoint_osd1_runtime': float(endpoint1_time),
        'current_full_runtime': float(trace.runtime[endpoint] + current_time),
        'tesseract_runtime': tess_runtime,
        **{f'endpoint_{key}': value for key, value in features.items()},
    }

    recon = {
        'shot_id': shot_id,
        'distance': context.distance,
        'p': context.p,
        'syndrome': intervention_gating_bits(syndrome),
        'endpoint': endpoint,
        'legacy_backend_endpoint': legacy_endpoint,
        'endpoint_llr_json': json.dumps(trace.llr[endpoint].tolist()),
        'legacy_backend_llr_json': json.dumps(legacy_llr.tolist()),
        'osd_order': order,
        'H_hash': sha256(context.H.tobytes()).hexdigest(),
        'prior_llr_hash': sha256(context.prior_llr.tobytes()).hexdigest(),
        'osd_method': 'osd_cs',
        'combination_sweep': intervention_gating_config['full_osd_combination_sweep'],
        'candidate_count': intervention_gating_candidate_count(context, order),
        'candidate_objective': 'minimum_hamming_weight',
        'tie_breaking': 'first_candidate_strict_improvement',
        'legacy_convention': 'raw_signed_argsort',
        'current_convention': 'benchmark_signed_prior_weighted_cs',
        'legacy_column_order': json.dumps(legacy_order.tolist()),
        'current_column_order': json.dumps(current_order.tolist()),
        'legacy_pivots': json.dumps(legacy_pivots.tolist()),
        'current_pivots': json.dumps(current_pivots.tolist()),
        'legacy_correction': intervention_gating_bits(legacy),
        'legacy_same_snapshot_correction': intervention_gating_bits(legacy_same_snapshot),
        'legacy_unified_correction': intervention_gating_bits(legacy_unified),
        'legacy_unified_column_order': json.dumps(legacy_unified_order.tolist()),
        'legacy_unified_pivots': json.dumps(legacy_unified_pivots.tolist()),
        'current_correction': intervention_gating_bits(current),
        'legacy_logical_label': legacy_out[3],
        'current_logical_label': current_out[3],
        'python_reference_audited': int(audit_reference),
        'legacy_backend_audited': int(audit_legacy_backend),
        'bp_endpoint_equal': bp_endpoint_equal,
        'bp_llr_allclose': bp_llr_allclose,
        'bp_llr_exact': bp_llr_exact,
        'bp_llr_max_abs_diff': bp_llr_max_abs_diff,
        'unified_input_correction_equal': int(np.array_equal(legacy_same_snapshot, current)),
        'controlled_column_order_equal': int(np.array_equal(legacy_order, legacy_unified_order)),
        'controlled_pivot_equal': int(np.array_equal(legacy_pivots, legacy_unified_pivots)),
        'controlled_correction_equal': int(np.array_equal(legacy, legacy_unified)),
        'legacy_unified_correct': legacy_unified_out[1],
        'unified_column_order_equal': int(np.array_equal(legacy_unified_order, current_order)),
        'unified_pivot_equal': int(np.array_equal(legacy_unified_pivots, current_pivots)),
        'unified_implementation_equal': int(np.array_equal(legacy_unified, current)),
        'legacy_correct': legacy_out[1],
        'current_correct': current_out[1],
        'ordering_equal': int(np.array_equal(legacy_order, current_order)),
        'pivot_equal': int(np.array_equal(legacy_pivots, current_pivots)),
        'correction_equal': int(np.array_equal(legacy, current)),
    }

    return (row, recon)

if intervention_gating_stage_cached([POPULATION_PATH, RECON_PATH, MISMATCH_PATH]):
    population = intervention_gating_read_csv(POPULATION_PATH)
    reconciliation_trials = intervention_gating_read_csv(RECON_PATH)
    reconciliation_mismatches = intervention_gating_read_csv(MISMATCH_PATH)
else:
    all_rows, recon_rows = ([], [])
    for intervention_gating_level in intervention_gating_config['levels']:
        for intervention_gating_p_index, intervention_gating_p in enumerate(intervention_gating_config['p_values']):
            intervention_gating_context = intervention_gating_context_for(intervention_gating_level, intervention_gating_p)
            point_rows = []
            intervention_gating_seen = 0

            while intervention_gating_seen < intervention_gating_config['maximum_shots_per_point']:
                intervention_gating_amount = min(intervention_gating_config['sampling_batch_size'], intervention_gating_config['maximum_shots_per_point'] - intervention_gating_seen)
                sampler = intervention_gating_context.dem.compile_sampler(seed=intervention_gating_config['random_seed'] + intervention_gating_level * 100003 + intervention_gating_p_index * 1009 + intervention_gating_seen)
                syndromes, truths, errors = sampler.sample(shots=intervention_gating_amount, return_errors=True)
                intervention_gating_tess, tess_runtime = intervention_gating_tesseract_predictions(intervention_gating_context, syndromes)

                for local, intervention_gating_values in enumerate(tqdm(zip(syndromes, truths, errors, intervention_gating_tess), total=intervention_gating_amount, desc=f'unbiased d={intervention_gating_context.distance} p={intervention_gating_p:g}', disable=intervention_gating_config['print_level'] == 1)):
                    intervention_gating_syndrome, intervention_gating_truth, intervention_gating_error, tess_prediction = [np.asarray(x, dtype=np.uint8) for x in intervention_gating_values]
                    intervention_gating_shot_id = f'L{intervention_gating_level}-p{intervention_gating_p_index}-b{intervention_gating_seen}-i{local}'
                    audit_reference = intervention_gating_seen + local < intervention_gating_config['reference_audit_shots_per_point']
                    audit_legacy_backend = intervention_gating_seen + local < intervention_gating_config['reconciliation_shots_per_point']
                    intervention_gating_row, recon = decoded_shot(intervention_gating_context, intervention_gating_syndrome, intervention_gating_truth, intervention_gating_error, tess_prediction, intervention_gating_shot_id, tess_runtime, audit_reference, audit_legacy_backend)
                    point_rows.append(intervention_gating_row)

                    if intervention_gating_seen + local < intervention_gating_config['reconciliation_shots_per_point']:
                        recon_rows.append(recon)

                intervention_gating_seen += intervention_gating_amount
                intervention_gating_point = pd.DataFrame(point_rows)
                unsafe = int((intervention_gating_point.standard_correct == 0).sum())
                intervention_gating_log(2, {
                    'stage': 'unbiased_population',
                    'distance': intervention_gating_context.distance,
                    'p': intervention_gating_p,
                    'shots': intervention_gating_seen,
                    'unsafe': unsafe,
                })

                if intervention_gating_seen >= intervention_gating_config['minimum_shots_per_point'] and unsafe >= intervention_gating_config['minimum_unsafe_events']:
                    break

            all_rows.extend(point_rows)

    population = pd.DataFrame(all_rows)
    reconciliation_trials = pd.DataFrame(recon_rows)
    reconciliation_mismatches = reconciliation_trials[reconciliation_trials.correction_equal == 0].head(20).copy()
    intervention_gating_save_csv(population, POPULATION_PATH)
    intervention_gating_save_csv(reconciliation_trials, RECON_PATH)
    intervention_gating_save_csv(reconciliation_mismatches, MISMATCH_PATH)

population['y_unsafe'] = 1 - population.standard_correct
population['stage2_role'] = [stage2_role_for(shot_id, split) for shot_id, split in zip(population.shot_id, population.split)]
population['y_gap'] = ((population.standard_correct == 0) & (population.tesseract_correct == 1)).astype(int)
population['y_benefit_full'] = ((population.standard_correct == 0) & (population.current_full_correct == 1)).astype(int)
population['y_regress_full'] = ((population.standard_correct == 1) & (population.current_full_correct == 0)).astype(int)

for utility_penalty in intervention_gating_config['utility_penalties']:
    population[f'utility_full_lambda{utility_penalty}'] = population.y_benefit_full - utility_penalty * population.y_regress_full
population['cohort'] = np.select([(population.standard_correct == 1) & (population.current_full_correct == 1) & (population.tesseract_correct == 1), (population.standard_correct == 0) & (population.current_full_correct == 1) & (population.tesseract_correct == 1), (population.standard_correct == 0) & (population.current_full_correct == 0) & (population.tesseract_correct == 1)], ['A', 'G', 'C'], default='other')
forced_osd_reconciliation = reconciliation_trials.groupby(['distance', 'p'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    ordering_match_rate=('ordering_equal', 'mean'),
    pivot_match_rate=('pivot_equal', 'mean'),
    correction_match_rate=('correction_equal', 'mean'),
    legacy_success=('legacy_correct', 'mean'),
    current_success=('current_correct', 'mean'),
    bp_endpoint_match_rate=('bp_endpoint_equal', 'mean'),
    bp_llr_match_rate=('bp_llr_allclose', 'mean'),
    bp_llr_exact_match_rate=('bp_llr_exact', 'mean'),
    max_bp_llr_abs_diff=('bp_llr_max_abs_diff', 'max'),
    controlled_ordering_match_rate=('controlled_column_order_equal', 'mean'),
    controlled_pivot_match_rate=('controlled_pivot_equal', 'mean'),
    controlled_correction_match_rate=('controlled_correction_equal', 'mean'),
    unified_ordering_match_rate=('unified_column_order_equal', 'mean'),
    unified_pivot_match_rate=('unified_pivot_equal', 'mean'),
    unified_implementation_match_rate=('unified_implementation_equal', 'mean'),
)

for intervention_gating_index, summary in forced_osd_reconciliation.iterrows():
    intervention_gating_block = reconciliation_trials[(reconciliation_trials.distance == summary.distance) & (reconciliation_trials.p == summary.p)]
    for legacy_value, current_value in product([0, 1], repeat=2):
        forced_osd_reconciliation.loc[intervention_gating_index, f'legacy{legacy_value}_current{current_value}'] = int(((intervention_gating_block.legacy_correct == legacy_value) & (intervention_gating_block.current_correct == current_value)).sum())
    closed = bool((intervention_gating_block.bp_endpoint_equal == 1).all() and (intervention_gating_block.bp_llr_exact == 1).all() and (intervention_gating_block.unified_column_order_equal == 1).all() and (intervention_gating_block.unified_pivot_equal == 1).all() and (intervention_gating_block.unified_implementation_equal == 1).all())
    forced_osd_reconciliation.loc[intervention_gating_index, 'reconciliation_status'] = 'closed_after_unified_convention' if closed else 'unresolved_backend_mismatch'

intervention_gating_save_csv(forced_osd_reconciliation, intervention_gating_data_paths.output('forced_osd_reconciliation_csv'))
label_summary = population.groupby(['distance', 'p', 'split'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    unsafe_rate=('y_unsafe', 'mean'),
    gap_rate=('y_gap', 'mean'),
    benefit_full_rate=('y_benefit_full', 'mean'),
    regression_full_rate=('y_regress_full', 'mean'),
    forced_full_success=('current_full_correct', 'mean'),
    tesseract_success=('tesseract_correct', 'mean'),
)
intervention_gating_save_csv(label_summary, intervention_gating_data_paths.output('intervention_label_summary_csv'))

if intervention_gating_config['print_level'] >= 2:
    display(forced_osd_reconciliation)


## Experiments 4–7: calibrated Stage-1 gate and cost-sensitive intervention

Calibrate the gate that decides whether to override standard BPOSD. Optimize using development data and include the cost of overriding an already correct baseline result.


In [ ]:
intervention_gating_w = intervention_gating_config['primary_window']
STATIC = ['endpoint_syndrome_consistent', 'endpoint_iteration', 'endpoint_llr_min', 'endpoint_llr_q10', 'endpoint_llr_q25', 'endpoint_llr_median', 'endpoint_llr_max', 'endpoint_llr_spread', 'endpoint_hard_prior_cost']
RANK = ['endpoint_rank_kendall', 'endpoint_rank_kendall_trend', 'endpoint_rank_jaccard', 'endpoint_rank_boundary_motion', f'endpoint_rank_persistence_w{intervention_gating_w}']
MESSAGE = ['endpoint_message_residual', 'endpoint_belief_movement', 'endpoint_sign_flip_rate', 'endpoint_extrinsic_conflict', f'endpoint_residual_mean_w{intervention_gating_w}', f'endpoint_residual_slope_w{intervention_gating_w}', f'endpoint_belief_mean_w{intervention_gating_w}', f'endpoint_flip_mean_w{intervention_gating_w}', f'endpoint_conflict_mean_w{intervention_gating_w}']
HISTORY = ['endpoint_history_min_residual', 'endpoint_history_max_rank_stability', 'endpoint_history_large_oscillation_episodes', 'endpoint_history_valid_entries', 'endpoint_history_time_since_first_valid']
intervention_gating_feature_sets = {
    'static': STATIC,
    'rank': STATIC + RANK,
    'message': STATIC + MESSAGE,
    'history': STATIC + HISTORY,
    'full': STATIC + RANK + MESSAGE + HISTORY,
}
FORBIDDEN = {'truth', 'error', 'standard_correct', 'current_full_correct', 'tesseract_correct', 'y_unsafe', 'y_gap'}
assert not FORBIDDEN & set(sum(intervention_gating_feature_sets.values(), []))

class CalibratedRisk:

    def __init__(
        self,
        base,
        calibrator,
        constant=None,
    ):
        self.base, self.calibrator, self.constant = (base, calibrator, constant)

    def predict(self, X):
        if self.constant is not None:
            return np.full(len(X), self.constant)
        raw = np.clip(self.base.predict_proba(X)[:, 1], 1e-06, 1 - 1e-06)

        return self.calibrator.predict_proba(logit(raw).reshape(-1, 1))[:, 1]

def base_model(kind):
    if kind == 'logistic':
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=intervention_gating_config['random_seed']))
    return DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=intervention_gating_config['random_seed'])

def fit_calibrated(
    frame,
    features,
    target,
    kind,
):
    fit = frame[frame.split == 'fit']
    calibration = frame[frame.split == 'calibration']
    if fit[target].nunique() < 2:
        return (CalibratedRisk(None, None, float(fit[target].mean() if len(fit) else 0.0)), 'constant')
    model = base_model(kind)

    model.fit(fit[features].fillna(0), fit[target])
    raw = np.clip(model.predict_proba(calibration[features].fillna(0))[:, 1], 1e-06, 1 - 1e-06)
    if calibration[target].nunique() < 2:
        return (CalibratedRisk(model, None, float(calibration[target].mean() if len(calibration) else fit[target].mean())), 'constant_calibration')
    calibrator = LogisticRegression(max_iter=1000)

    calibrator.fit(logit(raw).reshape(-1, 1), calibration[target])
    return (CalibratedRisk(model, calibrator), 'ok')

def prediction_metrics(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if len(labels) == 0:
        return {
            'auroc': np.nan,
            'auprc': np.nan,
            'brier': np.nan,
            'ece': np.nan,
        }

    auroc = roc_auc_score(labels, scores) if len(np.unique(labels)) == 2 else np.nan
    auprc = average_precision_score(labels, scores) if labels.sum() else np.nan
    bins = np.linspace(0, 1, intervention_gating_config['calibration_bins'] + 1)
    ids = np.clip(np.digitize(scores, bins) - 1, 0, len(bins) - 2)
    ece = sum((np.mean(ids == b) * abs(labels[ids == b].mean() - scores[ids == b].mean()) for b in np.unique(ids)))

    return {
        'auroc': auroc,
        'auprc': auprc,
        'brier': brier_score_loss(labels, scores),
        'ece': ece,
    }

GATE_MODELS = {}
gate_rows = []
calibration_rows = []

for intervention_gating_target, intervention_gating_feature_set, intervention_gating_kind in product(['y_unsafe', 'y_gap'], intervention_gating_feature_sets, ['logistic', 'tree']):
    intervention_gating_model, status = fit_calibrated(population, intervention_gating_feature_sets[intervention_gating_feature_set], intervention_gating_target, intervention_gating_kind)
    GATE_MODELS[intervention_gating_target, intervention_gating_feature_set, intervention_gating_kind] = intervention_gating_model
    test = population[population.split == 'test']

    intervention_gating_scores = intervention_gating_model.predict(test[intervention_gating_feature_sets[intervention_gating_feature_set]].fillna(0))
    intervention_gating_metrics = prediction_metrics(test[intervention_gating_target], intervention_gating_scores)
    gate_rows.append({
        'target': intervention_gating_target,
        'feature_set': intervention_gating_feature_set,
        'model': intervention_gating_kind,
        'status': status,
        **intervention_gating_metrics,
    })
    bins = np.linspace(0, 1, intervention_gating_config['calibration_bins'] + 1)
    ids = np.clip(np.digitize(intervention_gating_scores, bins) - 1, 0, len(bins) - 2)

    for b in np.unique(ids):
        calibration_rows.append({
            'target': intervention_gating_target,
            'feature_set': intervention_gating_feature_set,
            'model': intervention_gating_kind,
            'bin': int(b),
            'shots': int(np.sum(ids == b)),
            'mean_risk': float(intervention_gating_scores[ids == b].mean()),
            'observed_rate': float(test[intervention_gating_target].to_numpy()[ids == b].mean()),
        })

stage1_gate_ablation = pd.DataFrame(gate_rows)
stage1_calibration = pd.DataFrame(calibration_rows)
intervention_gating_save_csv(stage1_gate_ablation, intervention_gating_data_paths.output('stage1_gate_ablation_csv'))
intervention_gating_save_csv(stage1_calibration, intervention_gating_data_paths.output('stage1_calibration_csv'))

for intervention_gating_feature_set in intervention_gating_feature_sets:
    population[f'risk_{intervention_gating_feature_set}'] = GATE_MODELS['y_unsafe', intervention_gating_feature_set, 'logistic'].predict(population[intervention_gating_feature_sets[intervention_gating_feature_set]].fillna(0))

def intervention_outcome(block, trigger, recovery_column):
    standard = block.standard_correct.to_numpy(dtype=int)
    recovery = block[recovery_column].to_numpy(dtype=int)
    result = np.where(trigger, recovery, standard)
    benefit = trigger & (standard == 0) & (recovery == 1)
    regress = trigger & (standard == 1) & (recovery == 0)

    return (result, benefit, regress)

risk_rows = []
test = population[population.split == 'test'].copy()
percentiles = test.risk_full.rank(pct=True, method='average')

for intervention_gating_lower, intervention_gating_upper in zip(intervention_gating_config['risk_percentile_edges'][:-1], intervention_gating_config['risk_percentile_edges'][1:]):
    intervention_gating_selected = test[(percentiles > intervention_gating_lower) & (percentiles <= intervention_gating_upper)]
    benefit = (intervention_gating_selected.standard_correct == 0) & (intervention_gating_selected.current_full_correct == 1)
    regress = (intervention_gating_selected.standard_correct == 1) & (intervention_gating_selected.current_full_correct == 0)
    risk_rows.append({
        'percentile_low': intervention_gating_lower,
        'percentile_high': intervention_gating_upper,
        'shots': len(intervention_gating_selected),
        'mean_predicted_risk': intervention_gating_selected.risk_full.mean(),
        'unsafe_rate': intervention_gating_selected.y_unsafe.mean(),
        'benefit_rate': benefit.mean(),
        'regression_rate': regress.mean(),
        'benefit_regression_ratio': benefit.sum() / regress.sum() if regress.sum() else np.inf,
    })

risk_stratified = pd.DataFrame(risk_rows)
intervention_gating_save_csv(risk_stratified, intervention_gating_data_paths.output('risk_stratified_intervention_csv'))
frontier_rows = []

for split in ['calibration', 'test']:
    intervention_gating_block = population[population.split == split]
    for intervention_gating_feature_set, intervention_gating_threshold in product(intervention_gating_feature_sets, intervention_gating_config['gate_thresholds']):
        trigger = intervention_gating_block[f'risk_{intervention_gating_feature_set}'].to_numpy() >= intervention_gating_threshold
        result, benefit, regress = intervention_outcome(intervention_gating_block, trigger, 'current_full_correct')
        wrong = intervention_gating_block.standard_correct.to_numpy() == 0
        intervention_gating_correct = ~wrong
        intervention_gating_gap = wrong & (intervention_gating_block.tesseract_correct.to_numpy() == 1)
        frontier_rows.append({
            'split': split,
            'feature_set': intervention_gating_feature_set,
            'threshold': intervention_gating_threshold,
            'shots': len(intervention_gating_block),
            'intervention_rate': trigger.mean(),
            'LER': 1 - result.mean(),
            'delta_LER': 1 - result.mean() - (1 - intervention_gating_block.standard_correct.mean()),
            'repair_rate': benefit.sum() / wrong.sum() if wrong.sum() else np.nan,
            'gap_repair_rate': np.sum(trigger & intervention_gating_gap & (intervention_gating_block.current_full_correct.to_numpy() == 1)) / intervention_gating_gap.sum() if intervention_gating_gap.sum() else np.nan,
            'regression_rate': regress.sum() / intervention_gating_correct.sum() if intervention_gating_correct.sum() else np.nan,
        })

intervention_threshold_frontier = pd.DataFrame(frontier_rows)
intervention_gating_save_csv(intervention_threshold_frontier, intervention_gating_data_paths.output('intervention_threshold_frontier_csv'))
utility_rows = []
SELECTED_GATE = {}

for intervention_gating_feature_set, penalty in product(intervention_gating_feature_sets, intervention_gating_config['utility_penalties']):
    intervention_gating_candidates = intervention_threshold_frontier[(intervention_threshold_frontier.split == 'calibration') & (intervention_threshold_frontier.feature_set == intervention_gating_feature_set)].copy()
    calibration = population[population.split == 'calibration']
    utilities = []

    for intervention_gating_threshold in intervention_gating_candidates.threshold:
        trigger = calibration[f'risk_{intervention_gating_feature_set}'].to_numpy() >= intervention_gating_threshold
        intervention_gating_, benefit, regress = intervention_outcome(calibration, trigger, 'current_full_correct')
        utilities.append(int(benefit.sum()) - penalty * int(regress.sum()))

    intervention_gating_candidates['utility'] = utilities
    chosen = intervention_gating_candidates.sort_values(['utility', 'intervention_rate'], ascending=[False, True]).iloc[0]
    SELECTED_GATE[intervention_gating_feature_set, penalty] = float(chosen.threshold)
    test_row = intervention_threshold_frontier[(intervention_threshold_frontier.split == 'test') & (intervention_threshold_frontier.feature_set == intervention_gating_feature_set) & (intervention_threshold_frontier.threshold == chosen.threshold)].iloc[0]
    utility_rows.append({
        'feature_set': intervention_gating_feature_set,
        'penalty': penalty,
        'selected_threshold': chosen.threshold,
        'recovery_policy': 'endpoint_full_provisional',
        'development_utility': chosen.utility,
        **{f'test_{k}': test_row[k] for k in ['intervention_rate', 'LER', 'delta_LER', 'repair_rate', 'gap_repair_rate', 'regression_rate']},
    })

provisional_full_osd_gate = pd.DataFrame(utility_rows)
intervention_gating_save_csv(provisional_full_osd_gate, intervention_gating_data_paths.output('provisional_full_osd_gate_selection_csv'))
if intervention_gating_config['print_level'] >= 2:
    display(stage1_gate_ablation)


## Experiments 8–14: conditional Stage-2 snapshot selection and gated adaptive order

For permitted interventions, choose a stored snapshot and tested OSD order. Evaluate conditional recovery separately from gate quality, without truth or future snapshots as runtime inputs.


In [ ]:
SNAPSHOT_PATH = intervention_gating_data_paths.output('stage2_snapshot_features_csv')
penalty = intervention_gating_config['primary_utility_penalty']
for intervention_gating_feature_set in intervention_gating_feature_sets:
    population[f'candidate_gate_{intervention_gating_feature_set}'] = population[f'risk_{intervention_gating_feature_set}'] >= min(intervention_gating_config['gate_thresholds'])
population['stage2_candidate_union'] = population[[f'candidate_gate_{name}' for name in intervention_gating_feature_sets]].any(axis=1)
S2_STATIC = ['iteration', 'syndrome_consistent', 'syndrome_residual', 'llr_min', 'llr_q10', 'llr_q25', 'llr_median', 'llr_max', 'llr_spread']
S2_RANK = ['rank_kendall', 'rank_kendall_trend', 'rank_jaccard', 'rank_boundary_motion', f'rank_persistence_w{intervention_gating_w}']
S2_MESSAGE = ['message_residual', 'belief_movement', 'sign_flip_rate', 'extrinsic_conflict', f'residual_mean_w{intervention_gating_w}', f'residual_slope_w{intervention_gating_w}', f'belief_mean_w{intervention_gating_w}', f'flip_mean_w{intervention_gating_w}', f'conflict_mean_w{intervention_gating_w}']
S2_HISTORY = [name.removeprefix('endpoint_') for name in HISTORY]
S2_FEATURES = {
    'static': S2_STATIC,
    'rank': S2_STATIC + S2_RANK,
    'message': S2_STATIC + S2_MESSAGE,
    'history': S2_STATIC + S2_HISTORY,
    'full': S2_STATIC + S2_RANK + S2_MESSAGE + S2_HISTORY,
}

ALL_CAUSAL_COLUMNS = sorted(set(sum(S2_FEATURES.values(), []) + ['hard_prior_cost'] + [f'{stem}_w{window}' for window in intervention_gating_config['feature_windows'] for stem in ['residual_mean', 'residual_slope', 'belief_mean', 'flip_mean', 'conflict_mean', 'rank_persistence']]))
SNAPSHOT_COLUMNS = ['shot_id', 'level', 'distance', 'p', 'split', 'stage2_role', 'endpoint'] + ALL_CAUSAL_COLUMNS + ['osd0_correct', 'osd1_correct', 'osdfull_correct', 'osd0_runtime', 'osd1_runtime', 'osdfull_runtime']
if intervention_gating_stage_cached([SNAPSHOT_PATH]):
    stage2_snapshots = intervention_gating_read_csv(SNAPSHOT_PATH)
else:
    snapshot_rows = []
    intervention_gating_candidates = population[population.stage2_candidate_union & population.split.isin(['calibration', 'test'])]
    for intervention_gating_case in tqdm(intervention_gating_candidates.itertuples(index=False), total=len(intervention_gating_candidates), desc='Stage-2 snapshots', disable=intervention_gating_config['print_level'] == 1):
        intervention_gating_context = intervention_gating_context_for(int(intervention_gating_case.level), float(intervention_gating_case.p))
        intervention_gating_syndrome = intervention_gating_bit_array(intervention_gating_case.syndrome, intervention_gating_context.H.shape[0])
        intervention_gating_truth = intervention_gating_bit_array(intervention_gating_case.truth, intervention_gating_context.L.shape[0])
        intervention_gating_error = intervention_gating_bit_array(intervention_gating_case.error, intervention_gating_context.H.shape[1])
        intervention_gating_trace = intervention_gating_trace_min_sum(intervention_gating_context, intervention_gating_syndrome, intervention_gating_config['bp_max_iterations'])
        intervention_gating_endpoint = intervention_gating_endpoint_of(intervention_gating_trace)
        intervention_gating_order = intervention_gating_full_order(intervention_gating_context)

        for intervention_gating_t in range(1, intervention_gating_endpoint + 1):
            intervention_gating_outcomes = {}
            for intervention_gating_q in sorted({0, 1, intervention_gating_order}):
                intervention_gating_correction, intervention_gating_runtime, intervention_gating_, intervention_gating_ = osd_decode(intervention_gating_context, intervention_gating_syndrome, intervention_gating_trace.llr[intervention_gating_t], intervention_gating_q)
                intervention_gating_outcomes[intervention_gating_q] = (*intervention_gating_outcome(intervention_gating_context, intervention_gating_correction, intervention_gating_syndrome, intervention_gating_truth, intervention_gating_error), intervention_gating_runtime)

            snapshot_rows.append({
                'shot_id': intervention_gating_case.shot_id,
                'level': intervention_gating_case.level,
                'distance': intervention_gating_case.distance,
                'p': intervention_gating_case.p,
                'split': intervention_gating_case.split,
                'stage2_role': intervention_gating_case.stage2_role,
                'iteration': intervention_gating_t,
                'endpoint': intervention_gating_endpoint,
                **causal_features(intervention_gating_context, intervention_gating_trace, intervention_gating_syndrome, intervention_gating_t),
                'osd0_correct': intervention_gating_outcomes[0][1],
                'osd1_correct': intervention_gating_outcomes[1][1],
                'osdfull_correct': intervention_gating_outcomes[intervention_gating_order][1],
                'osd0_runtime': intervention_gating_outcomes[0][4],
                'osd1_runtime': intervention_gating_outcomes[1][4],
                'osdfull_runtime': intervention_gating_outcomes[intervention_gating_order][4],
            })

    stage2_snapshots = pd.DataFrame(snapshot_rows, columns=SNAPSHOT_COLUMNS)
    intervention_gating_save_csv(stage2_snapshots, SNAPSHOT_PATH)

STAGE2_BLOCKS = {shot_id: block.sort_values('iteration') for shot_id, block in stage2_snapshots.groupby('shot_id')}

class ConstantStage2:

    def __init__(self, probability):
        self.probability = float(probability)

    def predict_proba(self, X):
        probability = np.full(len(X), self.probability)
        return np.column_stack([1 - probability, probability])

def fit_stage2(
    frame,
    features,
    target='osd1_correct',
    kind='logistic',
):
    train = frame[frame.stage2_role == 'selector_fit']
    if len(train) == 0 or train[target].nunique() < 2:
        probability = float(train[target].mean()) if len(train) else 0.0
        return (ConstantStage2(probability), 'constant')
    model = base_model(kind)

    model.fit(train[features].fillna(0), train[target])
    return (model, 'ok')

def stage2_score(
    model,
    frame,
    features,
    target='osd1_correct',
):
    if len(frame) == 0:
        return np.empty(0, dtype=float)
    return model.predict_proba(frame[features].fillna(0))[:, 1]

primary_stage2_calibration_ids = set(population[(population.split == 'calibration') & population.candidate_gate_full].shot_id)
primary_stage2_test_ids = set(population[(population.split == 'test') & population.candidate_gate_full].shot_id)
primary_stage2_ids = primary_stage2_calibration_ids | primary_stage2_test_ids
STAGE2_MODELS = {}
STAGE2_ABLATION_MODELS = {}
selector_diagnostic_rows = []
selector_trial_rows = []

for intervention_gating_target, intervention_gating_feature_set, intervention_gating_kind in product(['osd0_correct', 'osd1_correct'], S2_FEATURES, ['logistic', 'tree']):
    common_frame = stage2_snapshots[stage2_snapshots.shot_id.isin(primary_stage2_calibration_ids)].copy()
    intervention_gating_model, status = fit_stage2(common_frame, S2_FEATURES[intervention_gating_feature_set], intervention_gating_target, intervention_gating_kind)
    STAGE2_ABLATION_MODELS[intervention_gating_target, intervention_gating_feature_set, intervention_gating_kind] = intervention_gating_model
    deployment_ids = set(population[(population.split == 'calibration') & population[f'candidate_gate_{intervention_gating_feature_set}']].shot_id)
    deployment_frame = stage2_snapshots[stage2_snapshots.shot_id.isin(deployment_ids)].copy()
    deployment_model, deployment_status = fit_stage2(deployment_frame, S2_FEATURES[intervention_gating_feature_set], intervention_gating_target, intervention_gating_kind)
    STAGE2_MODELS[intervention_gating_target, intervention_gating_feature_set, intervention_gating_kind] = deployment_model
    evaluated = stage2_snapshots[stage2_snapshots.shot_id.isin(primary_stage2_test_ids)]
    intervention_gating_scores = stage2_score(intervention_gating_model, evaluated, S2_FEATURES[intervention_gating_feature_set], intervention_gating_target)
    intervention_gating_metrics = prediction_metrics(evaluated[intervention_gating_target], intervention_gating_scores)
    selector_diagnostic_rows.append({
        'target': intervention_gating_target,
        'feature_set': intervention_gating_feature_set,
        'model': intervention_gating_kind,
        'status': status,
        'deployment_status': deployment_status,
        'unit': 'snapshot_row',
        **intervention_gating_metrics,
    })

    for intervention_gating_shot_id, intervention_gating_block in evaluated.groupby('shot_id'):
        intervention_gating_block = intervention_gating_block.sort_values('iteration')
        block_scores = stage2_score(intervention_gating_model, intervention_gating_block, S2_FEATURES[intervention_gating_feature_set], intervention_gating_target)
        intervention_gating_selected = intervention_gating_block.iloc[int(np.argmax(block_scores))]
        selector_trial_rows.append({
            'target': intervention_gating_target,
            'feature_set': intervention_gating_feature_set,
            'model': intervention_gating_kind,
            'status': status,
            'shot_id': intervention_gating_shot_id,
            'selected_iteration': int(intervention_gating_selected.iteration),
            'endpoint': int(intervention_gating_selected.endpoint),
            'success': int(intervention_gating_selected[intervention_gating_target]),
        })

stage2_snapshot_diagnostic = pd.DataFrame(selector_diagnostic_rows)
intervention_gating_save_csv(stage2_snapshot_diagnostic, intervention_gating_data_paths.output('stage2_snapshot_prediction_diagnostic_csv'))
stage2_selector_trials = pd.DataFrame(selector_trial_rows, columns=['target', 'feature_set', 'model', 'status', 'shot_id', 'selected_iteration', 'endpoint', 'success'])
intervention_gating_save_csv(stage2_selector_trials, intervention_gating_trace_dir / 'stage2_selector_trials.csv')
selector_aggregate = stage2_selector_trials.groupby(['target', 'feature_set', 'model', 'status'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    selection_success=('success', 'mean'),
    mean_selected_iteration=('selected_iteration', 'mean'),
    mean_endpoint=('endpoint', 'mean'),
)
selector_keys = stage2_snapshot_diagnostic[['target', 'feature_set', 'model', 'status']]
stage2_selector_ablation = selector_keys.merge(selector_aggregate, how='left', on=['target', 'feature_set', 'model', 'status'])
stage2_selector_ablation['shots'] = stage2_selector_ablation.shots.fillna(0).astype(int)
stage2_selector_ablation['unit'] = 'physical_shot'
intervention_gating_save_csv(stage2_selector_ablation, intervention_gating_data_paths.output('stage2_selector_ablation_csv'))
fixed_rows = []

for (split, intervention_gating_shot_id), intervention_gating_block in stage2_snapshots[stage2_snapshots.shot_id.isin(primary_stage2_ids)].groupby(['split', 'shot_id']):
    intervention_gating_block = intervention_gating_block.sort_values('iteration')
    intervention_gating_endpoint_success = int(intervention_gating_block.iloc[-1].osd1_correct)
    oracle = int(intervention_gating_block.osd1_correct.max())

    for intervention_gating_fixed in intervention_gating_config['fixed_snapshot_iterations']:
        intervention_gating_selected = min(intervention_gating_fixed, int(intervention_gating_block.endpoint.iloc[0]))
        success = int(intervention_gating_block.loc[intervention_gating_block.iteration == intervention_gating_selected, 'osd1_correct'].iloc[0])
        fixed_rows.append({
            'split': split,
            'stage2_role': intervention_gating_block.stage2_role.iloc[0],
            'shot_id': intervention_gating_shot_id,
            'fixed_iteration': intervention_gating_fixed,
            'selected_iteration': intervention_gating_selected,
            'success': success,
            'endpoint_success': intervention_gating_endpoint_success,
            'prefix_oracle_success': oracle,
        })

stage2_fixed_trials = pd.DataFrame(fixed_rows, columns=['split', 'stage2_role', 'shot_id', 'fixed_iteration', 'selected_iteration', 'success', 'endpoint_success', 'prefix_oracle_success'])
stage2_prefix_fixed = stage2_fixed_trials.groupby(['split', 'stage2_role', 'fixed_iteration'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    fixed_success=('success', 'mean'),
    endpoint_success=('endpoint_success', 'mean'),
    prefix_oracle_success=('prefix_oracle_success', 'mean'),
)
intervention_gating_save_csv(stage2_prefix_fixed, intervention_gating_data_paths.output('stage2_prefix_oracle_fixed_t_csv'))
calibration_fixed = stage2_prefix_fixed[stage2_prefix_fixed.stage2_role == 'policy_tune']
BEST_FIXED = int(calibration_fixed.sort_values(['fixed_success', 'fixed_iteration'], ascending=[False, True]).iloc[0].fixed_iteration) if len(calibration_fixed) else 1
fixed_test_by_feature = {}

for intervention_gating_feature_set in S2_FEATURES:
    fixed_outcomes = []
    for intervention_gating_, intervention_gating_block in stage2_snapshots[stage2_snapshots.shot_id.isin(primary_stage2_test_ids)].groupby('shot_id'):
        selected_iteration = min(BEST_FIXED, int(intervention_gating_block.endpoint.iloc[0]))
        fixed_outcomes.append(int(intervention_gating_block.loc[intervention_gating_block.iteration == selected_iteration, 'osd1_correct'].iloc[0]))

    fixed_test_by_feature[intervention_gating_feature_set] = np.mean(fixed_outcomes) if fixed_outcomes else np.nan

stage2_selector_ablation['best_fixed_osd1_success'] = stage2_selector_ablation.feature_set.map(fixed_test_by_feature)
stage2_selector_ablation['delta_vs_best_fixed'] = np.where(stage2_selector_ablation.target == 'osd1_correct', stage2_selector_ablation.selection_success - stage2_selector_ablation.best_fixed_osd1_success, np.nan)
intervention_gating_save_csv(stage2_selector_ablation, intervention_gating_data_paths.output('stage2_selector_ablation_csv'))
intervention_gating_shuffle_rows = []
intervention_gating_rng = np.random.default_rng(intervention_gating_config['random_seed'] + 77)

for intervention_gating_repetition in tqdm(range(intervention_gating_config['temporal_shuffle_repetitions']), desc='Stage-2 temporal shuffle', disable=intervention_gating_config['print_level'] == 1):
    intervention_gating_shuffled = stage2_snapshots.copy()
    dynamic = S2_RANK + S2_MESSAGE
    for intervention_gating_, intervention_gating_indices in intervention_gating_shuffled.groupby('shot_id').groups.items():
        intervention_gating_shuffled.loc[intervention_gating_indices, dynamic] = intervention_gating_shuffled.loc[intervention_gating_rng.permutation(intervention_gating_indices), dynamic].to_numpy()
    calibration_ids = set(population[(population.split == 'calibration') & population.candidate_gate_full].shot_id)
    test_ids = set(population[(population.split == 'test') & population.candidate_gate_full].shot_id)
    conditional = intervention_gating_shuffled[intervention_gating_shuffled.shot_id.isin(calibration_ids | test_ids)]
    intervention_gating_model, status = fit_stage2(conditional, S2_FEATURES['full'])
    test_frame = conditional[(conditional.split == 'test') & conditional.shot_id.isin(test_ids)]
    intervention_gating_scores = stage2_score(intervention_gating_model, test_frame, S2_FEATURES['full'])
    row_metrics = prediction_metrics(test_frame.osd1_correct, intervention_gating_scores)
    selected_outcomes = []
    selected_iterations = []

    for intervention_gating_, intervention_gating_block in test_frame.assign(score=intervention_gating_scores).groupby('shot_id'):
        intervention_gating_selected = intervention_gating_block.iloc[int(np.argmax(intervention_gating_block.score.to_numpy()))]
        selected_outcomes.append(int(intervention_gating_selected.osd1_correct))
        selected_iterations.append(int(intervention_gating_selected.iteration))

    intervention_gating_shuffle_rows.append({
        'repetition': intervention_gating_repetition,
        'status': status,
        'unit': 'physical_shot',
        'shots': len(selected_outcomes),
        'selection_success': np.mean(selected_outcomes) if selected_outcomes else np.nan,
        'mean_selected_iteration': np.mean(selected_iterations) if selected_iterations else np.nan,
        **{f'snapshot_{key}': value for key, value in row_metrics.items()},
    })

intervention_gating_save_csv(pd.DataFrame(intervention_gating_shuffle_rows), intervention_gating_data_paths.output('stage2_temporal_shuffle_csv'))
buffer_trials = []
for intervention_gating_feature_set in S2_FEATURES:
    intervention_gating_model = STAGE2_MODELS['osd1_correct', intervention_gating_feature_set, 'logistic']
    for (split, intervention_gating_shot_id), intervention_gating_block in stage2_snapshots.groupby(['split', 'shot_id']):
        intervention_gating_case = population.loc[population.shot_id == intervention_gating_shot_id].iloc[0]
        if not bool(intervention_gating_case[f'candidate_gate_{intervention_gating_feature_set}']):
            continue
        intervention_gating_block = intervention_gating_block.sort_values('iteration')
        all_scores = stage2_score(intervention_gating_model, intervention_gating_block, S2_FEATURES[intervention_gating_feature_set])

        for intervention_gating_size in intervention_gating_config['prefix_buffer_sizes']:
            intervention_gating_candidates = intervention_gating_block if intervention_gating_size == -1 else intervention_gating_block.tail(intervention_gating_size)
            intervention_gating_scores = all_scores[-len(intervention_gating_candidates):]
            intervention_gating_selected = intervention_gating_candidates.iloc[int(np.argmax(intervention_gating_scores))]
            intervention_gating_endpoint_success = int(intervention_gating_block.iloc[-1].osd1_correct)
            oracle = int(intervention_gating_block.osd1_correct.max())
            buffer_trials.append({
                'feature_set': intervention_gating_feature_set,
                'split': split,
                'stage2_role': intervention_gating_case.stage2_role,
                'shot_id': intervention_gating_shot_id,
                'buffer_size': intervention_gating_size,
                'selected_iteration': int(intervention_gating_selected.iteration),
                'success': int(intervention_gating_selected.osd1_correct),
                'endpoint_success': intervention_gating_endpoint_success,
                'oracle_success': oracle,
            })

intervention_gating_prefix_buffer_trials = pd.DataFrame(buffer_trials, columns=['feature_set', 'split', 'stage2_role', 'shot_id', 'buffer_size', 'selected_iteration', 'success', 'endpoint_success', 'oracle_success'])
prefix_buffer_recovery = intervention_gating_prefix_buffer_trials.groupby(['feature_set', 'split', 'stage2_role', 'buffer_size'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    success_rate=('success', 'mean'),
    endpoint_rate=('endpoint_success', 'mean'),
    oracle_rate=('oracle_success', 'mean'),
)
prefix_buffer_recovery['selector_recovery_fraction'] = np.where(prefix_buffer_recovery.oracle_rate > prefix_buffer_recovery.endpoint_rate, (prefix_buffer_recovery.success_rate - prefix_buffer_recovery.endpoint_rate) / (prefix_buffer_recovery.oracle_rate - prefix_buffer_recovery.endpoint_rate), np.nan)
intervention_gating_save_csv(prefix_buffer_recovery, intervention_gating_data_paths.output('prefix_buffer_recovery_csv'))
SELECTED_BUFFER = {}

for intervention_gating_feature_set in S2_FEATURES:
    choices = prefix_buffer_recovery[(prefix_buffer_recovery.feature_set == intervention_gating_feature_set) & (prefix_buffer_recovery.stage2_role == 'policy_tune')]
    if len(choices):
        choices = choices.assign(buffer_cost=np.where(choices.buffer_size == -1, np.inf, choices.buffer_size))
        SELECTED_BUFFER[intervention_gating_feature_set] = int(choices.sort_values(['success_rate', 'buffer_cost'], ascending=[False, True]).iloc[0].buffer_size)
    else:
        SELECTED_BUFFER[intervention_gating_feature_set] = 1

def selected_snapshot(shot_id, feature_set):
    assert shot_id in STAGE2_BLOCKS, f'missing prefix snapshots for gate-positive shot {shot_id}'
    block = STAGE2_BLOCKS[shot_id]
    size = SELECTED_BUFFER[feature_set]
    candidates = block if size == -1 else block.tail(size)
    model = STAGE2_MODELS['osd1_correct', feature_set, 'logistic']
    scores = stage2_score(model, candidates, S2_FEATURES[feature_set])

    return candidates.iloc[int(np.argmax(scores))]

DYNAMIC_RECOVERY = {feature_set: {} for feature_set in S2_FEATURES}
for intervention_gating_feature_set in S2_FEATURES:
    candidate_ids = set(population[population[f'candidate_gate_{intervention_gating_feature_set}']].shot_id)
    for intervention_gating_shot_id in candidate_ids & set(STAGE2_BLOCKS):
        DYNAMIC_RECOVERY[intervention_gating_feature_set][intervention_gating_shot_id] = int(selected_snapshot(intervention_gating_shot_id, intervention_gating_feature_set).osd1_correct)

def evaluate_deployed_gate(block, feature_set, threshold):
    trigger = block[f'risk_{feature_set}'].to_numpy() >= threshold
    standard = block.standard_correct.to_numpy(dtype=int)
    mapped = block.shot_id.map(DYNAMIC_RECOVERY[feature_set])
    assert mapped[trigger].notna().all(), 'missing dynamic recovery for a triggered candidate'
    recovery = np.where(trigger, mapped.fillna(pd.Series(standard, index=block.index)).to_numpy(dtype=int), standard)
    result = np.where(trigger, recovery, standard)
    benefit = trigger & (standard == 0) & (recovery == 1)
    regress = trigger & (standard == 1) & (recovery == 0)
    wrong = standard == 0
    correct = ~wrong
    gap = wrong & (block.tesseract_correct.to_numpy() == 1)

    return {
        'intervention_rate': trigger.mean(),
        'LER': 1 - result.mean(),
        'delta_LER': 1 - result.mean() - (1 - standard.mean()),
        'repair_rate': benefit.sum() / wrong.sum() if wrong.any() else np.nan,
        'gap_repair_rate': np.sum(trigger & gap & (recovery == 1)) / gap.sum() if gap.any() else np.nan,
        'regression_rate': regress.sum() / correct.sum() if correct.any() else np.nan,
        'benefits': int(benefit.sum()),
        'regressions': int(regress.sum()),
    }

SELECTED_GATE = {}
deployed_gate_rows = []
deployed_frontier_rows = []
policy_tune = population[population.stage2_role == 'policy_tune']
held_out = population[population.split == 'test']

for intervention_gating_feature_set, utility_penalty in product(S2_FEATURES, intervention_gating_config['utility_penalties']):
    choices = []
    for intervention_gating_threshold in intervention_gating_config['gate_thresholds']:
        intervention_gating_metrics = evaluate_deployed_gate(policy_tune, intervention_gating_feature_set, intervention_gating_threshold)
        utility = intervention_gating_metrics['benefits'] - utility_penalty * intervention_gating_metrics['regressions']
        choices.append({
            'threshold': intervention_gating_threshold,
            'utility': utility,
            **intervention_gating_metrics,
        })
        deployed_frontier_rows.append({
            'split': 'policy_tune',
            'feature_set': intervention_gating_feature_set,
            'penalty': utility_penalty,
            'threshold': intervention_gating_threshold,
            'utility': utility,
            **intervention_gating_metrics,
        })
        test_frontier_metrics = evaluate_deployed_gate(held_out, intervention_gating_feature_set, intervention_gating_threshold)
        deployed_frontier_rows.append({
            'split': 'test',
            'feature_set': intervention_gating_feature_set,
            'penalty': utility_penalty,
            'threshold': intervention_gating_threshold,
            'utility': np.nan,
            **test_frontier_metrics,
        })

    chosen = pd.DataFrame(choices).sort_values(['utility', 'intervention_rate'], ascending=[False, True]).iloc[0]
    SELECTED_GATE[intervention_gating_feature_set, utility_penalty] = float(chosen.threshold)
    test_metrics = evaluate_deployed_gate(held_out, intervention_gating_feature_set, float(chosen.threshold))
    deployed_gate_rows.append({
        'feature_set': intervention_gating_feature_set,
        'penalty': utility_penalty,
        'selected_threshold': chosen.threshold,
        'recovery_policy': 'dynamic_buffer_OSD1',
        'development_utility': chosen.utility,
        **{f'test_{key}': value for key, value in test_metrics.items() if key not in {'benefits', 'regressions'}},
    })

cost_sensitive_gate = pd.DataFrame(deployed_gate_rows)
intervention_gating_save_csv(cost_sensitive_gate, intervention_gating_data_paths.output('cost_sensitive_gate_selection_csv'))
deployed_gate_frontier = pd.DataFrame(deployed_frontier_rows)
intervention_gating_save_csv(deployed_gate_frontier, intervention_gating_data_paths.output('deployed_two_stage_gate_frontier_csv'))

for intervention_gating_feature_set in S2_FEATURES:
    population[f'gate_{intervention_gating_feature_set}'] = population[f'risk_{intervention_gating_feature_set}'] >= SELECTED_GATE[intervention_gating_feature_set, penalty]
population['stage2_union'] = population[[f'gate_{name}' for name in S2_FEATURES]].any(axis=1)
adaptive_trial_rows = []

for intervention_gating_feature_set in S2_FEATURES:
    for intervention_gating_case in population[population[f'gate_{intervention_gating_feature_set}'] & population.split.isin(['calibration', 'test'])].itertuples(index=False):
        intervention_gating_selected = selected_snapshot(intervention_gating_case.shot_id, intervention_gating_feature_set)
        one = pd.DataFrame([intervention_gating_selected])
        intervention_gating_p0 = stage2_score(STAGE2_MODELS['osd0_correct', intervention_gating_feature_set, 'logistic'], one, S2_FEATURES[intervention_gating_feature_set], 'osd0_correct')[0]
        intervention_gating_p1 = stage2_score(STAGE2_MODELS['osd1_correct', intervention_gating_feature_set, 'logistic'], one, S2_FEATURES[intervention_gating_feature_set], 'osd1_correct')[0]
        intervention_gating_context = intervention_gating_context_for(int(intervention_gating_case.level), float(intervention_gating_case.p))

        for threshold0, threshold1 in product(intervention_gating_config['adaptive_order_thresholds'], repeat=2):
            if intervention_gating_p0 >= threshold0:
                intervention_gating_order, osd_tier = (0, 'osd0')
            elif intervention_gating_p1 >= threshold1:
                intervention_gating_order, osd_tier = (1, 'osd1')
            else:
                intervention_gating_order, osd_tier = (intervention_gating_full_order(intervention_gating_context), 'full')

            intervention_gating_correct = int(intervention_gating_selected.osd0_correct if intervention_gating_order == 0 else intervention_gating_selected.osd1_correct if intervention_gating_order == 1 else intervention_gating_selected.osdfull_correct)
            runtime_column = 'osd0_runtime' if intervention_gating_order == 0 else 'osd1_runtime' if intervention_gating_order == 1 else 'osdfull_runtime'
            intervention_gating_osd_runtime = float(intervention_gating_selected[runtime_column])
            adaptive_trial_rows.append({
                'feature_set': intervention_gating_feature_set,
                'split': intervention_gating_case.split,
                'shot_id': intervention_gating_case.shot_id,
                'stage2_role': intervention_gating_case.stage2_role,
                'threshold0': threshold0,
                'threshold1': threshold1,
                'osd_order': intervention_gating_order,
                'osd_tier': osd_tier,
                'correct': intervention_gating_correct,
                'standard_correct': int(intervention_gating_case.standard_correct),
                'candidate_count': intervention_gating_candidate_count(intervention_gating_context, intervention_gating_order),
                'osd_runtime': intervention_gating_osd_runtime,
                'total_runtime': float(intervention_gating_case.endpoint_bp_runtime + intervention_gating_osd_runtime),
            })

intervention_gating_adaptive_trials = pd.DataFrame(adaptive_trial_rows, columns=['feature_set', 'split', 'stage2_role', 'shot_id', 'threshold0', 'threshold1', 'osd_order', 'osd_tier', 'correct', 'standard_correct', 'candidate_count', 'osd_runtime', 'total_runtime'])
gated_adaptive_order = intervention_gating_adaptive_trials.groupby(['feature_set', 'split', 'stage2_role', 'threshold0', 'threshold1'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    success_rate=('correct', 'mean'),
    mean_order=('osd_order', 'mean'),
    fraction_osd0=('osd_tier', lambda x: np.mean(x == 'osd0')),
    fraction_osd1=('osd_tier', lambda x: np.mean(x == 'osd1')),
    fraction_full=('osd_tier', lambda x: np.mean(x == 'full')),
    mean_candidate_count=('candidate_count', 'mean'),
    mean_osd_runtime=('osd_runtime', 'mean'),
    mean_total_runtime=('total_runtime', 'mean'),
    p95_total_runtime=('total_runtime', lambda x: x.quantile(0.95)),
)
gated_adaptive_order['repair_rate'] = np.nan
gated_adaptive_order['regression_rate'] = np.nan

for intervention_gating_index, summary in gated_adaptive_order.iterrows():
    intervention_gating_block = intervention_gating_adaptive_trials[(intervention_gating_adaptive_trials.feature_set == summary.feature_set) & (intervention_gating_adaptive_trials.split == summary.split) & (intervention_gating_adaptive_trials.threshold0 == summary.threshold0) & (intervention_gating_adaptive_trials.stage2_role == summary.stage2_role) & (intervention_gating_adaptive_trials.threshold1 == summary.threshold1)]
    wrong = intervention_gating_block.standard_correct == 0
    intervention_gating_correct = ~wrong
    gated_adaptive_order.loc[intervention_gating_index, 'repair_rate'] = intervention_gating_block.loc[wrong, 'correct'].mean() if wrong.any() else np.nan
    gated_adaptive_order.loc[intervention_gating_index, 'regression_rate'] = (1 - intervention_gating_block.loc[intervention_gating_correct, 'correct']).mean() if intervention_gating_correct.any() else np.nan

intervention_gating_save_csv(gated_adaptive_order, intervention_gating_data_paths.output('gated_adaptive_osd_order_csv'))
SELECTED_ADAPTIVE = {}
for intervention_gating_feature_set in S2_FEATURES:
    choices = gated_adaptive_order[(gated_adaptive_order.feature_set == intervention_gating_feature_set) & (gated_adaptive_order.stage2_role == 'policy_tune')]
    if len(choices):
        intervention_gating_row = choices.sort_values(['success_rate', 'mean_candidate_count'], ascending=[False, True]).iloc[0]
        SELECTED_ADAPTIVE[intervention_gating_feature_set] = (float(intervention_gating_row.threshold0), float(intervention_gating_row.threshold1))
    else:
        SELECTED_ADAPTIVE[intervention_gating_feature_set] = (0.5, 0.5)

recovery_rows = []
for intervention_gating_feature_set in S2_FEATURES:
    relevant = population[(population.split == 'test') & population[f'gate_{intervention_gating_feature_set}']]
    for policy in ['endpoint_OSD0', 'endpoint_OSD1', 'endpoint_full', 'best_fixed_OSD1', 'dynamic_buffer_OSD1', 'prefix_oracle_OSD1']:
        intervention_gating_outcomes = []
        for intervention_gating_case in relevant.itertuples(index=False):
            assert intervention_gating_case.shot_id in STAGE2_BLOCKS, f'missing prefix snapshots for gate-positive shot {intervention_gating_case.shot_id}'
            intervention_gating_block = STAGE2_BLOCKS[intervention_gating_case.shot_id]
            if policy == 'endpoint_OSD0':
                intervention_gating_value = intervention_gating_block.iloc[-1].osd0_correct
            elif policy == 'endpoint_OSD1':
                intervention_gating_value = intervention_gating_block.iloc[-1].osd1_correct
            elif policy == 'endpoint_full':
                intervention_gating_value = intervention_gating_block.iloc[-1].osdfull_correct
            elif policy == 'best_fixed_OSD1':
                intervention_gating_value = intervention_gating_block.loc[intervention_gating_block.iteration == min(BEST_FIXED, int(intervention_gating_block.endpoint.iloc[0])), 'osd1_correct'].iloc[0]
            elif policy == 'prefix_oracle_OSD1':
                intervention_gating_value = intervention_gating_block.osd1_correct.max()
            else:
                intervention_gating_value = selected_snapshot(intervention_gating_case.shot_id, intervention_gating_feature_set).osd1_correct

            intervention_gating_outcomes.append(int(intervention_gating_value))

        recovery_rows.append({
            'feature_set': intervention_gating_feature_set,
            'policy': policy,
            'shots': len(intervention_gating_outcomes),
            'conditional_success': np.mean(intervention_gating_outcomes) if intervention_gating_outcomes else np.nan,
        })

recovery_policy_comparison = pd.DataFrame(recovery_rows)
intervention_gating_save_csv(recovery_policy_comparison, intervention_gating_data_paths.output('recovery_policy_comparison_csv'))


## Experiments 15–17 and final held-out paired benchmark

Freeze the combined gate and recovery rule before the held-out paired comparison. Whole-population LER, repairs, regressions and cost determine whether the complete policy helps.


In [ ]:
def intervention_gating_paired_gap_interval(standard_correct, reference_correct):
    difference = 1 - np.asarray(standard_correct, float) - (1 - np.asarray(reference_correct, float))
    rng = np.random.default_rng(intervention_gating_config['random_seed'] + 303)
    boot = [rng.choice(difference, len(difference), replace=True).mean() for _ in range(intervention_gating_config['bootstrap_repetitions'])]

    return (float(difference.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975)))

def paired_ler_intervals(method_correct, standard_correct, seed_offset):
    method_wrong = 1 - np.asarray(method_correct, dtype=float)
    standard_wrong = 1 - np.asarray(standard_correct, dtype=float)
    assert len(method_wrong) == len(standard_wrong) and len(method_wrong)
    rng = np.random.default_rng(intervention_gating_config['random_seed'] + seed_offset)
    indices = rng.integers(0, len(method_wrong), size=(intervention_gating_config['bootstrap_repetitions'], len(method_wrong)))
    ler_boot = method_wrong[indices].mean(axis=1)
    delta_boot = (method_wrong - standard_wrong)[indices].mean(axis=1)

    return (float(method_wrong.mean()), float(np.quantile(ler_boot, 0.025)), float(np.quantile(ler_boot, 0.975)), float((method_wrong - standard_wrong).mean()), float(np.quantile(delta_boot, 0.025)), float(np.quantile(delta_boot, 0.975)))

def two_stage_result(
    case,
    feature_set,
    adaptive=False,
    fixed=False,
):
    if not bool(case[f'gate_{feature_set}']):
        standard_order = -1 if int(case.first_valid) >= 1 else intervention_gating_full_order(intervention_gating_context_for(int(case.level), float(case.p)))
        return (int(case.standard_correct), 0, standard_order, float(case.standard_runtime), 'none' if standard_order < 0 else 'full')
    assert case.shot_id in STAGE2_BLOCKS, f'missing prefix snapshots for gate-positive shot {case.shot_id}'
    block = STAGE2_BLOCKS[case.shot_id]

    if fixed:
        selected = block.loc[block.iteration == min(BEST_FIXED, int(block.endpoint.iloc[0]))].iloc[0]
    else:
        selected = selected_snapshot(case.shot_id, feature_set)
    order, osd_tier = (1, 'osd1')

    if adaptive:
        t0, t1 = SELECTED_ADAPTIVE[feature_set]
        one = pd.DataFrame([selected])
        p0 = stage2_score(STAGE2_MODELS['osd0_correct', feature_set, 'logistic'], one, S2_FEATURES[feature_set], 'osd0_correct')[0]
        p1 = stage2_score(STAGE2_MODELS['osd1_correct', feature_set, 'logistic'], one, S2_FEATURES[feature_set], 'osd1_correct')[0]
        context = intervention_gating_context_for(int(case.level), float(case.p))

        if p0 >= t0:
            order, osd_tier = (0, 'osd0')
        elif p1 >= t1:
            order, osd_tier = (1, 'osd1')
        else:
            order, osd_tier = (intervention_gating_full_order(context), 'full')

    correct = int(selected.osd0_correct if order == 0 else selected.osd1_correct if order == 1 else selected.osdfull_correct)
    runtime_column = 'osd0_runtime' if order == 0 else 'osd1_runtime' if order == 1 else 'osdfull_runtime'

    return (correct, 1, order, float(case.endpoint_bp_runtime + getattr(selected, runtime_column)), osd_tier)

endpoint_transition_rows = []
for intervention_gating_case in population.itertuples(index=False):
    for intervention_gating_order_name in ['osd0', 'osd1', 'full']:
        endpoint_value = int(getattr(intervention_gating_case, f'endpoint_{intervention_gating_order_name}_correct') if intervention_gating_order_name != 'full' else intervention_gating_case.current_full_correct)
        final_value = int(getattr(intervention_gating_case, f'final_{intervention_gating_order_name}_correct'))
        endpoint_transition_rows.append({
            'distance': intervention_gating_case.distance,
            'p': intervention_gating_case.p,
            'cohort': intervention_gating_case.cohort,
            'shot_id': intervention_gating_case.shot_id,
            'osd_order': intervention_gating_order_name,
            'endpoint_correct': endpoint_value,
            'final_correct': final_value,
            'transition': f'{endpoint_value}->{final_value}',
        })

endpoint_transitions = pd.DataFrame(endpoint_transition_rows)
endpoint_vs_final = endpoint_transitions.groupby(['distance', 'p', 'cohort', 'osd_order', 'transition'], as_index=False).agg(shots=('shot_id', 'nunique'))
intervention_gating_save_csv(endpoint_vs_final, intervention_gating_data_paths.output('endpoint_vs_final_transitions_csv'))
intervention_gating_parameter_scan_path = intervention_gating_data_paths.output('bp_parameter_two_stage_scan_csv')
intervention_gating_parameter_scan_cache = intervention_gating_stage_cached([intervention_gating_parameter_scan_path])
parameter_rows = []

if intervention_gating_parameter_scan_cache:
    intervention_gating_scan_cases = population.head(0)
else:
    intervention_gating_scan_cases = population[population.split == 'calibration'].groupby(['distance', 'p'], group_keys=False).head(intervention_gating_config['parameter_scan_case_limit'])

for intervention_gating_case in tqdm(intervention_gating_scan_cases.itertuples(index=False), total=len(intervention_gating_scan_cases), desc='two-stage BP parameter scan', disable=intervention_gating_config['print_level'] == 1):
    intervention_gating_context = intervention_gating_context_for(int(intervention_gating_case.level), float(intervention_gating_case.p))
    intervention_gating_syndrome = intervention_gating_bit_array(intervention_gating_case.syndrome, intervention_gating_context.H.shape[0])
    intervention_gating_truth = intervention_gating_bit_array(intervention_gating_case.truth, intervention_gating_context.L.shape[0])

    for intervention_gating_scaling, intervention_gating_damping in product(intervention_gating_config['scaling_values'], intervention_gating_config['damping_values']):
        intervention_gating_trace = intervention_gating_trace_min_sum(intervention_gating_context, intervention_gating_syndrome, intervention_gating_config['bp_max_iterations'], intervention_gating_scaling, intervention_gating_damping)
        intervention_gating_endpoint = intervention_gating_endpoint_of(intervention_gating_trace)
        endpoint_features = causal_features(intervention_gating_context, intervention_gating_trace, intervention_gating_syndrome, intervention_gating_endpoint)
        feature_row = pd.DataFrame([{f'endpoint_{k}': v for k, v in endpoint_features.items()}])

        risk = GATE_MODELS['y_unsafe', 'full', 'logistic'].predict(feature_row[intervention_gating_feature_sets['full']].fillna(0))[0]
        intervention_gating_successes = []
        scan_snapshot_rows = []

        for intervention_gating_t in range(1, intervention_gating_endpoint + 1):
            intervention_gating_correction, intervention_gating_, intervention_gating_, intervention_gating_ = osd_decode(intervention_gating_context, intervention_gating_syndrome, intervention_gating_trace.llr[intervention_gating_t], 1)
            intervention_gating_successes.append(intervention_gating_outcome(intervention_gating_context, intervention_gating_correction, intervention_gating_syndrome, intervention_gating_truth)[1])
            scan_snapshot_rows.append(causal_features(intervention_gating_context, intervention_gating_trace, intervention_gating_syndrome, intervention_gating_t))

        scan_snapshot_frame = pd.DataFrame(scan_snapshot_rows)
        buffer_size = SELECTED_BUFFER['full']
        scan_candidates = scan_snapshot_frame if buffer_size == -1 else scan_snapshot_frame.tail(buffer_size)
        scan_scores = stage2_score(STAGE2_MODELS['osd1_correct', 'full', 'logistic'], scan_candidates, S2_FEATURES['full'])
        dynamic_iteration = int(scan_candidates.iloc[int(np.argmax(scan_scores))].iteration)
        dynamic_ok = int(intervention_gating_successes[dynamic_iteration - 1])
        fixed_t = min(BEST_FIXED, intervention_gating_endpoint)
        fixed_correction, intervention_gating_, intervention_gating_, intervention_gating_ = osd_decode(intervention_gating_context, intervention_gating_syndrome, intervention_gating_trace.llr[fixed_t], 1)
        intervention_gating_standard = intervention_gating_trace.hard[intervention_gating_endpoint] if intervention_gating_trace.valid[intervention_gating_endpoint] else osd_decode(intervention_gating_context, intervention_gating_syndrome, intervention_gating_trace.llr[intervention_gating_endpoint], intervention_gating_full_order(intervention_gating_context), 'legacy_raw_signed')[0]
        standard_ok = intervention_gating_outcome(intervention_gating_context, intervention_gating_standard, intervention_gating_syndrome, intervention_gating_truth)[1]
        intervention = risk >= SELECTED_GATE['full', penalty]
        two_stage_ok = dynamic_ok if intervention else standard_ok
        parameter_rows.append({
            'shot_id': intervention_gating_case.shot_id,
            'distance': intervention_gating_case.distance,
            'p': intervention_gating_case.p,
            'scaling': intervention_gating_scaling,
            'damping': intervention_gating_damping,
            'population_scope': 'unbiased_calibration_subset',
            'policy_fit': 'baseline_frozen',
            'standard_correct': standard_ok,
            'unsafe_risk': risk,
            'prefix_oracle_osd1': max(intervention_gating_successes),
            'fixed_osd1_correct': intervention_gating_outcome(intervention_gating_context, fixed_correction, intervention_gating_syndrome, intervention_gating_truth)[1],
            'dynamic_selected_iteration': dynamic_iteration,
            'two_stage_correct': two_stage_ok,
        })

if intervention_gating_parameter_scan_cache:
    bp_parameter_scan = intervention_gating_read_csv(intervention_gating_parameter_scan_path)
else:
    parameter_trials = pd.DataFrame(parameter_rows)
    bp_parameter_scan = parameter_trials.groupby(['distance', 'p', 'scaling', 'damping', 'population_scope', 'policy_fit'], as_index=False).agg(
        shots=('shot_id', 'nunique'),
        standard_LER=('standard_correct', lambda x: 1 - x.mean()),
        mean_predicted_unsafe_risk=('unsafe_risk', 'mean'),
        prefix_oracle_osd1=('prefix_oracle_osd1', 'mean'),
        fixed_osd1_success=('fixed_osd1_correct', 'mean'),
        two_stage_LER=('two_stage_correct', lambda x: 1 - x.mean()),
    ) if len(parameter_trials) else pd.DataFrame(columns=['distance', 'p', 'scaling', 'damping', 'population_scope', 'policy_fit', 'shots', 'standard_LER', 'mean_predicted_unsafe_risk', 'prefix_oracle_osd1', 'fixed_osd1_success', 'two_stage_LER'])
    for intervention_gating_name in ['unsafe_auroc', 'unsafe_auprc', 'unsafe_brier', 'unsafe_ece']:
        bp_parameter_scan[intervention_gating_name] = np.nan

    for intervention_gating_index, summary in bp_parameter_scan.iterrows():
        intervention_gating_block = parameter_trials[(parameter_trials.distance == summary.distance) & (parameter_trials.p == summary.p) & (parameter_trials.scaling == summary.scaling) & (parameter_trials.damping == summary.damping)]
        intervention_gating_metrics = prediction_metrics(1 - intervention_gating_block.standard_correct.to_numpy(), intervention_gating_block.unsafe_risk.to_numpy())
        for intervention_gating_key, intervention_gating_value in intervention_gating_metrics.items():
            bp_parameter_scan.loc[intervention_gating_index, f'unsafe_{intervention_gating_key}'] = intervention_gating_value

    intervention_gating_save_csv(bp_parameter_scan, intervention_gating_parameter_scan_path)

FINAL_TRIAL_PATH = intervention_gating_trace_dir / 'two_stage_decoder_trials.csv'
if intervention_gating_stage_cached([FINAL_TRIAL_PATH]):
    final_trials = intervention_gating_read_csv(FINAL_TRIAL_PATH)
else:
    intervention_gating_final_rows = []
    for intervention_gating_, intervention_gating_case in tqdm(population[population.split == 'test'].iterrows(), total=(population.split == 'test').sum(), desc='final two-stage benchmark', disable=intervention_gating_config['print_level'] == 1):
        intervention_gating_standard_order = -1 if int(intervention_gating_case.first_valid) >= 1 else intervention_gating_full_order(intervention_gating_context_for(int(intervention_gating_case.level), float(intervention_gating_case.p)))
        baselines = {
            'standard_BPOSD': (intervention_gating_case.standard_correct, 0, intervention_gating_standard_order, intervention_gating_case.standard_runtime, 'none' if intervention_gating_standard_order < 0 else 'full'),
            'Tesseract': (intervention_gating_case.tesseract_correct, 0, -1, intervention_gating_case.tesseract_runtime, 'none'),
            'forced_endpoint_OSD1': (intervention_gating_case.endpoint_osd1_correct, 1, 1, intervention_gating_case.endpoint_bp_runtime + intervention_gating_case.endpoint_osd1_runtime, 'osd1'),
            'forced_endpoint_OSDfull': (intervention_gating_case.current_full_correct, 1, intervention_gating_full_order(intervention_gating_context_for(int(intervention_gating_case.level), float(intervention_gating_case.p))), intervention_gating_case.current_full_runtime, 'full'),
        }

        for intervention_gating_decoder, intervention_gating_values in baselines.items():
            intervention_gating_final_rows.append({
                'shot_id': intervention_gating_case.shot_id,
                'distance': intervention_gating_case.distance,
                'p': intervention_gating_case.p,
                'decoder': intervention_gating_decoder,
                'correct': int(intervention_gating_values[0]),
                'intervened': int(intervention_gating_values[1]),
                'osd_order': int(intervention_gating_values[2]),
                'osd_tier': intervention_gating_values[4],
                'runtime': intervention_gating_values[3],
            })

        fixed_result = two_stage_result(intervention_gating_case, 'full', fixed=True)
        intervention_gating_final_rows.append({
            'shot_id': intervention_gating_case.shot_id,
            'distance': intervention_gating_case.distance,
            'p': intervention_gating_case.p,
            'decoder': 'best_fixed_selective_OSD1',
            'correct': fixed_result[0],
            'intervened': fixed_result[1],
            'osd_order': fixed_result[2],
            'osd_tier': fixed_result[4],
            'runtime': fixed_result[3],
        })

        for intervention_gating_feature_set in ['static', 'rank', 'message', 'full']:
            result = two_stage_result(intervention_gating_case, intervention_gating_feature_set)
            intervention_gating_final_rows.append({
                'shot_id': intervention_gating_case.shot_id,
                'distance': intervention_gating_case.distance,
                'p': intervention_gating_case.p,
                'decoder': f'two_stage_{intervention_gating_feature_set}',
                'correct': result[0],
                'intervened': result[1],
                'osd_order': result[2],
                'osd_tier': result[4],
                'runtime': result[3],
            })

        adaptive = two_stage_result(intervention_gating_case, 'full', adaptive=True)
        intervention_gating_final_rows.append({
            'shot_id': intervention_gating_case.shot_id,
            'distance': intervention_gating_case.distance,
            'p': intervention_gating_case.p,
            'decoder': 'two_stage_adaptive_order',
            'correct': adaptive[0],
            'intervened': adaptive[1],
            'osd_order': adaptive[2],
            'osd_tier': adaptive[4],
            'runtime': adaptive[3],
        })

    final_trials = pd.DataFrame(intervention_gating_final_rows)
    intervention_gating_save_csv(final_trials, FINAL_TRIAL_PATH)

intervention_gating_benchmark_rows = []
for (intervention_gating_distance, intervention_gating_p), intervention_gating_point in final_trials.groupby(['distance', 'p']):
    intervention_gating_standard = intervention_gating_point[intervention_gating_point.decoder == 'standard_BPOSD'].set_index('shot_id').correct
    intervention_gating_tess = intervention_gating_point[intervention_gating_point.decoder == 'Tesseract'].set_index('shot_id').correct
    intervention_gating_gap, gap_low, gap_high = intervention_gating_paired_gap_interval(intervention_gating_standard, intervention_gating_tess)
    intervention_gating_gap_status = 'resolved' if gap_low > 0 else 'unresolved'

    for intervention_gating_decoder, intervention_gating_block in intervention_gating_point.groupby('decoder'):
        intervention_gating_values = intervention_gating_block.set_index('shot_id')
        base = intervention_gating_standard.loc[intervention_gating_values.index]
        reference = intervention_gating_tess.loc[intervention_gating_values.index]
        wrong = base == 0
        gap_mask = wrong & (reference == 1)
        correct_base = base == 1
        interval = paired_ler_intervals(intervention_gating_values.correct, base, int.from_bytes(sha256(str(intervention_gating_decoder).encode()).digest()[:2], 'little'))
        intervention_gating_ler, ler_low, ler_high, delta_ler, delta_low, delta_high = interval
        intervention_gating_benchmark_rows.append({
            'distance': intervention_gating_distance,
            'p': intervention_gating_p,
            'decoder': intervention_gating_decoder,
            'shots': len(intervention_gating_values),
            'LER': intervention_gating_ler,
            'LER_ci_low': ler_low,
            'LER_ci_high': ler_high,
            'delta_LER_vs_standard': delta_ler,
            'delta_LER_ci_low': delta_low,
            'delta_LER_ci_high': delta_high,
            'standard_LER': 1 - intervention_gating_standard.mean(),
            'Tesseract_LER': 1 - intervention_gating_tess.mean(),
            'paired_gap': intervention_gating_gap,
            'paired_gap_ci_low': gap_low,
            'paired_gap_ci_high': gap_high,
            'gap_status': intervention_gating_gap_status,
            'gap_closure': (1 - intervention_gating_standard.mean() - intervention_gating_ler) / intervention_gating_gap if intervention_gating_gap_status == 'resolved' else np.nan,
            'repair_rate': intervention_gating_values.loc[wrong, 'correct'].mean() if wrong.any() else np.nan,
            'gap_repair_rate': intervention_gating_values.loc[gap_mask, 'correct'].mean() if gap_mask.any() else np.nan,
            'regression_rate': (1 - intervention_gating_values.loc[correct_base, 'correct']).mean() if correct_base.any() else np.nan,
            'intervention_rate': intervention_gating_values.intervened.mean(),
            'fraction_osd0': np.mean(intervention_gating_values.osd_tier == 'osd0'),
            'fraction_osd1': np.mean(intervention_gating_values.osd_tier == 'osd1'),
            'fraction_full': np.mean(intervention_gating_values.osd_tier == 'full'),
            'mean_candidate_count': np.mean([0 if q < 0 else intervention_gating_candidate_count(intervention_gating_context_for(2 if intervention_gating_distance == 5 else 3, intervention_gating_p), q) for q in intervention_gating_values.osd_order]),
            'runtime_scope': 'batch_average_decode_kernel' if intervention_gating_decoder == 'Tesseract' else 'bp_and_osd_kernel_only',
            'runtime_excludes_feature_model': int(intervention_gating_decoder.startswith('two_stage') or intervention_gating_decoder.startswith('best_fixed')),
            'mean_runtime': intervention_gating_values.runtime.mean(),
            'p95_runtime': np.nan if intervention_gating_decoder == 'Tesseract' else intervention_gating_values.runtime.quantile(0.95),
        })

final_benchmark = pd.DataFrame(intervention_gating_benchmark_rows)
intervention_gating_save_csv(final_benchmark, intervention_gating_data_paths.output('final_two_stage_benchmark_csv'))
instability_ablation = final_benchmark[final_benchmark.decoder.isin(['two_stage_static', 'two_stage_rank', 'two_stage_message', 'two_stage_full'])].copy()
intervention_gating_save_csv(instability_ablation, intervention_gating_data_paths.output('two_stage_instability_ablation_csv'))
accuracy_cost = final_benchmark[['distance', 'p', 'decoder', 'LER', 'gap_closure', 'intervention_rate', 'fraction_full', 'mean_candidate_count', 'runtime_scope', 'runtime_excludes_feature_model', 'mean_runtime']].copy()
intervention_gating_save_csv(accuracy_cost, intervention_gating_data_paths.output('accuracy_cost_frontier_csv'))

if intervention_gating_config['print_level'] >= 2:
    display(final_benchmark)


## Figures and completeness checks



In [ ]:
intervention_gating_required_stats = ['forced_osd_reconciliation.csv', 'intervention_label_summary.csv', 'stage1_gate_ablation.csv', 'stage1_calibration.csv', 'risk_stratified_intervention.csv', 'intervention_threshold_frontier.csv', 'provisional_full_osd_gate_selection.csv', 'deployed_two_stage_gate_frontier.csv', 'cost_sensitive_gate_selection.csv', 'stage2_prefix_oracle_fixed_t.csv', 'stage2_selector_ablation.csv', 'stage2_snapshot_prediction_diagnostic.csv', 'stage2_temporal_shuffle.csv', 'prefix_buffer_recovery.csv', 'recovery_policy_comparison.csv', 'gated_adaptive_osd_order.csv', 'two_stage_instability_ablation.csv', 'bp_parameter_two_stage_scan.csv', 'endpoint_vs_final_transitions.csv', 'final_two_stage_benchmark.csv', 'accuracy_cost_frontier.csv']
intervention_gating_required_trace = ['forced_osd_reconciliation_trials.csv', 'forced_osd_mismatches.csv', 'unbiased_intervention_cases.csv', 'stage2_snapshot_features.csv', 'stage2_selector_trials.csv', 'two_stage_decoder_trials.csv']
assert all((intervention_gating_data_paths.output('name', name=name).is_file() for name in intervention_gating_required_stats))
assert all(((intervention_gating_trace_dir / name).is_file() for name in intervention_gating_required_trace))

intervention_gating_fig, intervention_gating_axes = plt.subplots(2, 3, figsize=(17, 9))
sns.heatmap(pd.crosstab(reconciliation_trials.legacy_correct, reconciliation_trials.current_correct), annot=True, fmt='d', ax=intervention_gating_axes[0, 0])
intervention_gating_axes[0, 0].set_title('Figure 1: forced-OSD reconciliation')

if len(stage1_calibration):
    calibration_plot = stage1_calibration[(stage1_calibration.target == 'y_unsafe') & (stage1_calibration.feature_set == 'full') & (stage1_calibration.model == 'logistic')]
    intervention_gating_axes[0, 1].plot(calibration_plot.mean_risk, calibration_plot.observed_rate, marker='o')
    intervention_gating_axes[0, 1].plot([0, 1], [0, 1], '--')
intervention_gating_axes[0, 1].set_title('Figure 2: intervention-risk calibration')
front = deployed_gate_frontier[(deployed_gate_frontier.split == 'test') & (deployed_gate_frontier.feature_set == 'full') & (deployed_gate_frontier.penalty == intervention_gating_config['primary_utility_penalty'])]

intervention_gating_axes[0, 2].plot(front.regression_rate, front.repair_rate, marker='o')
intervention_gating_axes[0, 2].set_title('Figure 3: repair vs regression')
test_fixed = stage2_prefix_fixed[stage2_prefix_fixed.split == 'test']

if len(test_fixed):
    intervention_gating_axes[1, 0].plot(test_fixed.fixed_iteration, test_fixed.fixed_success, marker='o', label='fixed t')
    intervention_gating_axes[1, 0].axhline(test_fixed.endpoint_success.iloc[0], label='endpoint')
    intervention_gating_axes[1, 0].axhline(test_fixed.prefix_oracle_success.iloc[0], label='oracle')
    dynamic = stage2_selector_ablation[(stage2_selector_ablation.target == 'osd1_correct') & (stage2_selector_ablation.feature_set == 'full') & (stage2_selector_ablation.model == 'logistic')]

    if len(dynamic) and dynamic.selection_success.notna().any():
        intervention_gating_axes[1, 0].scatter([BEST_FIXED], [dynamic.selection_success.iloc[0]], marker='*', s=140, label='dynamic')
    best_fixed_test = test_fixed[test_fixed.fixed_iteration == BEST_FIXED]

    if len(best_fixed_test):
        intervention_gating_axes[1, 0].scatter([BEST_FIXED], [best_fixed_test.fixed_success.iloc[0]], marker='s', label='best fixed')
    intervention_gating_axes[1, 0].legend()

intervention_gating_axes[1, 0].set_title('Figure 4: Stage-2 snapshot policies')
if len(instability_ablation):
    sns.barplot(instability_ablation, x='decoder', y='LER', ax=intervention_gating_axes[1, 1])
    intervention_gating_axes[1, 1].tick_params(axis='x', rotation=30)
intervention_gating_axes[1, 1].set_title('Figure 5: instability ablation')

if len(accuracy_cost):
    sns.scatterplot(
        accuracy_cost,
        x='mean_candidate_count',
        y='LER',
        hue='decoder',
        ax=intervention_gating_axes[1, 2],
    )
intervention_gating_axes[1, 2].set_title('Figure 6: final accuracy-cost')
plt.tight_layout()

plt.show()
stage2_selector_fit_shots = int(population[(population.stage2_role == 'selector_fit') & population.candidate_gate_full].shot_id.nunique())
stage2_policy_tune_shots = int(population[(population.stage2_role == 'policy_tune') & population.candidate_gate_full].shot_id.nunique())
stage2_test_shots = len(primary_stage2_test_ids)
reconciliation_closed = bool((forced_osd_reconciliation.reconciliation_status == 'closed_after_unified_convention').all())
experiment_status = 'blocked_unresolved_forced_osd_reconciliation' if not reconciliation_closed else 'complete' if stage2_selector_fit_shots and stage2_policy_tune_shots and stage2_test_shots else 'insufficient_stage2_events'
status = pd.DataFrame([{
    'status': experiment_status,
    'distance_5_primary': True,
    'distance_9_status': 'enabled' if 3 in intervention_gating_config['levels'] else 'disabled',
    'legacy_current_mismatches': int((reconciliation_trials.correction_equal == 0).sum()),
    'forced_osd_reconciliation_status': ';'.join(sorted(forced_osd_reconciliation.reconciliation_status.unique())),
    'downstream_results_provisional': int(not reconciliation_closed),
    'stage2_selector_fit_shots': stage2_selector_fit_shots,
    'stage2_policy_tune_shots': stage2_policy_tune_shots,
    'stage2_test_shots': stage2_test_shots,
    'stage1_primary_threshold': SELECTED_GATE['full', intervention_gating_config['primary_utility_penalty']],
    'stage2_best_fixed_iteration': BEST_FIXED,
    'stage2_selected_buffer': SELECTED_BUFFER['full'],
}])
intervention_gating_save_csv(status, intervention_gating_data_paths.output('experiment_status_csv'))
intervention_gating_log(2, status.iloc[0].to_dict())
